# 01. Análise exploratória (EDA)

**Objetivo:** diagnosticar as quatro bases brutas (estrutura, nulos, duplicatas, outliers, distribuições e comportamento no tempo) sem usar inadimplência. Aqui só se diagnostica; o tratamento fica para outro notebook.

In [ ]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

import src.functions as fn #Funcoes criadas no projeto
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Leitura das bases e criação de cópias para tratamento.
cadastral_bruta = pd.read_parquet("../dados/base_cadastral.parquet")
cadastral = cadastral_bruta.copy()
submissao_bruta = pd.read_parquet("../dados/base_submissao.parquet")
submissao = submissao_bruta.copy()
emprestimos_bruta = pd.read_parquet("../dados/historico_emprestimos.parquet")
emprestimos = emprestimos_bruta.copy()
parcelas_bruta = pd.read_parquet("../dados/historico_parcelas.parquet")
parcelas = parcelas_bruta.copy()

for nome, df in [("cadastral", cadastral_bruta), ("submissao", submissao_bruta),
                 ("emprestimos", emprestimos_bruta), ("parcelas", parcelas_bruta)]:
    print(f"{nome}: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

## Base Cadastral

In [ ]:
cadastral.info()

In [ ]:
cadastral.head()

In [ ]:
# Estado de partida.
print(cadastral.shape)
print("nulos por coluna:")
nulos = cadastral.isna().sum()
print(nulos[nulos > 0])
print("linhas duplicadas:", cadastral.duplicated().sum())
print("id_cliente únicos:", cadastral["id_cliente"].nunique())

# Categorias de cada coluna a padronizar, na grafia original.
for col in ["sexo", "tipo_renda", "possui_carro", "possui_imovel"]:
    print(f"\n{col}")
    print(cadastral[col].value_counts(dropna=False))

In [ ]:
# Aplicando função criada para fazer uma auditoria profunda em dados categóricos (excluindo colunas com tipos incorretos).
resumo, freq, variantes = fn.auditar_categoricas(
    cadastral, excluir=["data_nascimento", "renda_anual"]
)
resumo

In [ ]:
freq

In [ ]:
variantes

In [ ]:
cadastral.describe()

In [ ]:
# Avaliando estrutura de colunas que deveriam ter tipos diferents:
print(cadastral["data_nascimento"].str.replace(r"\d", "9", regex=True).value_counts(dropna=False).head(10))
print(cadastral["renda_anual"].str.replace(r"\d", "", regex=True).value_counts(dropna=False).head(10))
cadastral[["data_nascimento", "renda_anual"]].sample(8, random_state=42)

### Decisões para `cadastral` (variáveis categóricas)

- `sexo`, `tipo_renda`, `possui_carro`, `possui_imovel`: padronizar caixa, acentos e espaços para um rótulo canônico (Feminino, Masculino, Sim, Não, Assalariado, Autônomo).
- Placeholder "Não informado" em `sexo`, `possui_carro` e `possui_imovel` (~0,1% cada): converter para nulo e decidir a imputação no pipeline, ajustada só no treino.
- `nivel_educacao`, `tipo_moradia`, `ocupacao` (categorias): sem variantes de digitação.
- Pendentes: tratamento do placeholder em `tipo_renda`, `tipo_organizacao` e `estado_civil`; significado dos nulos de `ocupacao`; categorias raras.
- Há colunas que deverão ser excluídas antes do merge com as demais, como `sexo` (dados sensíveis) e `qtd_filhos` (redundante)

### Decisões de tipos para `cadastral`

- `data_nascimento`: dois formatos, `AAAA-MM-DD` (50.562) e `DD/MM/AAAA` (38, ordem confirmada porque o 1º campo passa de 12). Converter com `["%Y-%m-%d", "%d/%m/%Y"]`.
- `renda_anual`: texto com ponto de milhar e vírgula decimal, 101 valores com prefixo `R$` e 101 com "não informado" (viram nulo). Converter com `decimal=","` e `milhar="."`.
- `renda_anual`: 67 valores negativos, a tratar como inconsistência depois da conversão.
- `qtd_membros_familia`: float, converter para `Int64` se não houver valores fracionários.

In [ ]:
# Padroniza e converte placeholders em nulo.
cadastral, rel_a = fn.padronizar_categoricas(
    cadastral_bruta,
    ["sexo", "possui_carro", "possui_imovel", "estado_civil",
     "ocupacao", "tipo_organizacao", "tipo_renda"],
)
print(rel_a)
print("linhas:", len(cadastral))
for col in ["sexo", "tipo_renda", "possui_carro", "possui_imovel",
            "estado_civil", "tipo_organizacao"]:
    print(f"\n{col}")
    print(cadastral[col].value_counts(dropna=False))

In [ ]:
# Converte só a data de nascimento; o restante do cadastral segue como está.
cadastral, rel_data = fn.converter_tipos(
    cadastral, datas={"data_nascimento": ["%Y-%m-%d", "%d/%m/%Y"]}
)

print(rel_data)
print("dtype:", cadastral["data_nascimento"].dtype)
print("período:", cadastral["data_nascimento"].min(), "->", cadastral["data_nascimento"].max())
print("linhas:", len(cadastral))

In [ ]:
falhou = cadastral["data_nascimento"].isna()
print(cadastral_bruta.loc[falhou, "data_nascimento"].value_counts().head(10))

In [ ]:
referencia = pd.Timestamp("2025-02-24")
idade = (referencia - cadastral["data_nascimento"]).dt.days / 365

print(idade.describe())
print("nascimento no futuro:", (cadastral["data_nascimento"] > referencia).sum())
print("idade < 18:", (idade < 18).sum())
print("idade > 100:", (idade > 100).sum())
print("datas mais extremas (menores):")
print(cadastral["data_nascimento"].nsmallest(5).tolist())
print("datas mais extremas (maiores):")
print(cadastral["data_nascimento"].nlargest(5).tolist())

In [ ]:
nasc = cadastral["data_nascimento"]
bruto = cadastral_bruta["data_nascimento"].astype("string")

# Frequência das datas mais repetidas: sentinelas aparecem com contagem anormal.
print(nasc.value_counts().head(5))

# Menores de 18 que NÃO são a sentinela de 2030, para ver o que sobra.
idade = (pd.Timestamp("2025-02-14") - nasc).dt.days / 365.25
sentinelas = pd.to_datetime(["1901-05-20", "2030-01-01"])
menores_reais = (idade < 18) & ~nasc.isin(sentinelas)
print("menores de 18 fora das sentinelas:", menores_reais.sum())
print(idade[menores_reais].describe())
print(nasc[menores_reais].value_counts().head(5))

# Quantos registros brutos usam cada sentinela.
print("brutos 31/02/1990:", (bruto == "31/02/1990").sum())
print("1901-05-20:", nasc.eq(sentinelas[0]).sum(), "| 2030-01-01:", nasc.eq(sentinelas[1]).sum())

In [ ]:
sentinelas = pd.to_datetime(["1901-05-20", "2030-01-01", "2012-07-15"])
nasc = cadastral["data_nascimento"]

# Flag: data impossível (nula após a conversão), ou uma das 3 sentinelas.
cadastral["data_nascimento_invalida"] = np.where(
    nasc.isna() | nasc.isin(sentinelas), 1, 0
)

# Sentinelas viram nulo; as datas reais permanecem.
cadastral["data_nascimento"] = nasc.mask(nasc.isin(sentinelas))

# Checagens.
print("flag = 1:", cadastral["data_nascimento_invalida"].sum())
print("nulos:", cadastral["data_nascimento"].isna().sum())
print("flag coincide com nulo:",
      (cadastral["data_nascimento_invalida"] == 1).equals(cadastral["data_nascimento"].isna()))
print("período:", cadastral["data_nascimento"].min(), "->", cadastral["data_nascimento"].max())
print("linhas:", len(cadastral))

In [ ]:
cadastral, rel_renda = fn.converter_tipos(
    cadastral, numericas_texto={"renda_anual": {"decimal": ",", "milhar": "."}}
)

print(rel_renda)
print(cadastral["renda_anual"].describe())
print("negativos:", (cadastral["renda_anual"] < 0).sum())
print("zeros:", (cadastral["renda_anual"] == 0).sum())

In [ ]:
renda = cadastral["renda_anual"]

# Negativos: o valor absoluto cabe na distribuição?
print(renda[renda < 0].abs().describe())

# Zeros por tipo de renda.
print(cadastral.loc[renda == 0, "tipo_renda"].value_counts())

# Cauda superior: valores mais altos e quantos passam de limiares.
print(renda.nlargest(10).tolist())
for limite in [1_000_000, 5_000_000, 10_000_000]:
    print(f"> {limite:,}:", (renda > limite).sum())

# Percentis para localizar a descontinuidade.
print(renda.quantile([0.99, 0.999, 0.9999]))

In [ ]:
renda = cadastral["renda_anual"]
neg = renda < 0
zero = renda == 0
print("negativos:", neg.sum(), "| zeros:", zero.sum())
print(cadastral.loc[neg, "tipo_renda"].value_counts())
print(cadastral.loc[zero, "tipo_renda"].value_counts())

# Zeros e negativos coincidem com outras anomalias?
print("zeros também nulos em tipo_renda:", cadastral.loc[renda == 0, "tipo_renda"].value_counts())
print("negativos e zeros nas mesmas linhas:", ((renda < 0) & (renda == 0)).sum())

# Faixas da cauda, em escala log, para ver onde existe um vazio.
faixas = pd.cut(renda[renda >= 500_000],
                bins=[5e5, 1e6, 2e6, 5e6, 1e7, 2e7, 5e7, 1e9])
print(faixas.value_counts().sort_index())

# Se dividirmos por 100, os valores altos caem na distribuição normal?
alto = renda[renda >= 5_000_000]
print((alto / 100).describe())
print(renda.describe())
print("tipo_renda dos altos:\n", cadastral.loc[alto.index, "tipo_renda"].value_counts())

### Pendente: `renda_anual` (aguardando resposta do cliente)

Conversão: 101 valores ("não informado") viraram nulo; 50.499 valores numéricos.

Observações:
- 67 valores negativos (magnitude entre 2.133 e 49.832) e 67 zeros. A coincidência de contagem precisa ser verificada.
- Mediana 148.500; 99% dos valores positivos abaixo de 1 milhão (referência: 50.263 valores entre 16.500 e 990.000).
- Cauda: 102 valores acima de 1 milhão e 71 acima de 5 milhões, com máximo de 117 milhões. Percentis: p99 = 540 mil, p99,9 = 15,3 milhões, p99,99 = 32,8 milhões.
- Faixas: 486 valores entre 500 mil e 1 milhão, 31 entre 1 e 5 milhões, 61 acima de 10 milhões. Diferente do `valor_credito`, não há vazio nítido, e a cauda é gradual.
- Ao dividir por 100 os 71 valores acima de 5 milhões, a mediana vai a ~204 mil e o máximo a 1,17 milhão, acima do máximo da referência (990 mil). A hipótese de escala 100× não se confirma como no crédito.
- Os valores altos aparecem em assalariado (33), associado comercial (19), aposentado (15) e servidor público (4), sem autônomo.

Decisões em aberto:
1. Tratamento de negativos e zeros (nulo, flag ou manter).
2. Critério de corte da cauda para o pipeline. Opção em análise: regra calculável (ex.: p99 × 10), em vez de limite fixo, para escalar com novos clientes. Com os dados atuais, p99 × 10 = 5,4 milhões, quase o corte de 5 milhões.
3. Revisão da correção de `valor_credito` em `submissao` (63 casos divididos por 100). Um pedido muito acima do valor do bem pode ser sinal legítimo de risco, e não erro. Os originais estão preservados em `valor_credito_original` e `credito_corrigido`, então a decisão é reversível.

Perguntas ao cliente: o que significam renda zero e negativa? Valores acima de 1 milhão são plausíveis? Existe alguma escala ou unidade diferente por origem de dados?

In [ ]:
renda = cadastral["renda_anual"]
renda_pos = renda[renda > 0]
idade = (pd.Timestamp("2025-02-24") - cadastral["data_nascimento"]).dt.days / 365.25

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

sns.histplot(np.log10(renda_pos), bins=60, ax=axes[0, 0])
axes[0, 0].set_title("renda_anual (log10, só positivos)")

sns.boxplot(x=np.log10(renda_pos), ax=axes[0, 1])
axes[0, 1].set_title("renda_anual (log10)")

sns.histplot(idade.dropna(), bins=50, ax=axes[0, 2])
axes[0, 2].set_title("idade (anos)")

sns.countplot(x=cadastral["qtd_membros_familia"], ax=axes[1, 0])
axes[1, 0].set_title("qtd_membros_familia")

sns.countplot(x=cadastral["nota_regiao_cliente"], ax=axes[1, 1])
axes[1, 1].set_title("nota_regiao_cliente")

sns.countplot(x=cadastral["nota_regiao_cliente_cidade"], ax=axes[1, 2])
axes[1, 2].set_title("nota_regiao_cliente_cidade")

plt.tight_layout()
plt.show()

print(cadastral[["qtd_filhos", "qtd_membros_familia",
                 "nota_regiao_cliente", "nota_regiao_cliente_cidade"]].describe())

In [ ]:
# Análise de membros da família e filhos
membros = cadastral["qtd_membros_familia"]
filhos = cadastral["qtd_filhos"]

print("membros == 0:", (membros == 0).sum())
print("membros < filhos + 1:", (membros < filhos + 1).sum())
print("membros < filhos:", (membros < filhos).sum())
print(pd.crosstab(filhos.clip(upper=5), membros.clip(upper=8)))
print("membros > 8:", (membros > 8).sum(), "| filhos > 5:", (filhos > 5).sum())
print("correlação entre notas:",
      cadastral["nota_regiao_cliente"].corr(cadastral["nota_regiao_cliente_cidade"], method="spearman"))
print(pd.crosstab(cadastral["nota_regiao_cliente"], cadastral["nota_regiao_cliente_cidade"]))

In [ ]:
# Tratamento de valores nulos e inconsistentes em `qtd_membros_familia`. A decisão é imputar nulo para 0
cadastral["qtd_membros_familia"] = (
    cadastral["qtd_membros_familia"].replace(0, np.nan).astype("Int64")
)
print("nulos:", cadastral["qtd_membros_familia"].isna().sum())
print(cadastral["qtd_membros_familia"].dtype)
print("linhas:", len(cadastral))

In [ ]:
# Análise de valores nulos e inconsistentes em colunas categóricas
for col in ["estado_civil", "ocupacao", "tipo_organizacao"]:
    print(f"\n{col}")
    print(cadastral[col].value_counts(dropna=False).to_string())

nulo_ocup = cadastral["ocupacao"].isna()
print("\nnulos de ocupacao por tipo_renda (% dentro de cada tipo):")
print(pd.crosstab(cadastral["tipo_renda"], nulo_ocup, normalize="index").round(3))
print("\ncontagem absoluta:")
print(pd.crosstab(cadastral["tipo_renda"], nulo_ocup))

In [ ]:
# Separa a 1ª e a 2ª ocorrência dos IDs repetidos em cadastral preservando a ordem original da tabela.
dup_cad_1 = cadastral.loc[cadastral.duplicated("id_cliente", keep="last")].set_index("id_cliente").sort_index()
dup_cad_2 = cadastral.loc[cadastral.duplicated("id_cliente", keep="first")].set_index("id_cliente").sort_index()

# Contabiliza em quantos pares cada coluna diverge entre a 1ª e a 2ª ocorrência.
dif_cad = (dup_cad_1.ne(dup_cad_2) & ~(dup_cad_1.isna() & dup_cad_2.isna())).sum()

print(f"Clientes repetidos: {len(dup_cad_1)} ({len(dup_cad_1) * 2} linhas)")
print(f"2ª ocorrência está 100% concentrada em index >= 50000? {np.array_equal(cadastral.index >= 50000, cadastral.duplicated('id_cliente', keep='first'))}")
print(f"Renda com centavos (!= .00) -> 1ª ocorrência: {(dup_cad_1['renda_anual'] % 1 != 0).sum()} | 2ª ocorrência: {(dup_cad_2['renda_anual'] % 1 != 0).sum()}")
print("\nColunas que divergem entre a 1ª e a 2ª ocorrência:")
display(dif_cad.loc[dif_cad > 0])

In [ ]:
# 1. Seleciona colunas textuais/categóricas de cadastral (exceto datas)
cols_cat_cad = [
    c for c in cadastral.select_dtypes(include=["object", "string", "category"]).columns 
    if c != "data_nascimento"
]

# 2. Aplica padronização em cópia isolada de cadastral
cad_teste, rel_cad = fn.padronizar_categoricas(cadastral, colunas=cols_cat_cad)

# 3. Separa 1ª e 2ª ocorrências alinhadas por id_cliente
cad_teste_1 = cad_teste.loc[cad_teste.duplicated("id_cliente", keep="last")].set_index("id_cliente").sort_index()
cad_teste_2 = cad_teste.loc[cad_teste.duplicated("id_cliente", keep="first")].set_index("id_cliente").sort_index()

# 4. Matriz de transição para estado_civil (onde houve 150 divergências)
mascara_ec = cad_teste_1["estado_civil"] != cad_teste_2["estado_civil"]
transicoes_ec = pd.crosstab(
    cad_teste_1.loc[mascara_ec, "estado_civil"],
    cad_teste_2.loc[mascara_ec, "estado_civil"],
    rownames=["1a_ocorrencia (<50k)"],
    colnames=["2a_ocorrencia (>=50k)"],
    dropna=False
)
print("--- Transições em estado_civil pós-padronização ---")
display(transicoes_ec)

# 5. Inspeciona divergências demográficas (sexo, data_nascimento, qtd_filhos, bens)
cols_demo = ["sexo", "data_nascimento", "qtd_filhos", "possui_carro", "possui_imovel"]
mascara_demo = (cad_teste_1[cols_demo].ne(cad_teste_2[cols_demo]) & ~(cad_teste_1[cols_demo].isna() & cad_teste_2[cols_demo].isna())).any(axis=1)

print("\n--- Casos com divergência demográfica ou patrimonial em cadastral ---")
display(pd.concat([
    cad_teste_1.loc[mascara_demo, cols_demo].add_suffix("_1a"),
    cad_teste_2.loc[mascara_demo, cols_demo].add_suffix("_2a")
], axis=1))

### Decisão: padronização de `sexo`, `tipo_renda`, `possui_carro` e `possui_imovel`

Problema: variantes de caixa, acento e espaço (6 grafias em `sexo`, 7 em `possui_carro` e `possui_imovel`, 13 em `tipo_renda`) e o placeholder "Não informado".
Regra: normalizar com `fn.padronizar_categoricas` (minúsculas, sem acentos, sem espaços sobrando). Em `sexo`, `possui_carro` e `possui_imovel`, o placeholder vira nulo. Em `tipo_renda`, o placeholder é mantido como categoria `nao informado`, pois a decisão está pendente.
Resultado: 2 categorias em `sexo`, `possui_carro` e `possui_imovel`, com 51, 49 e 54 nulos; 10 categorias em `tipo_renda`; 50.600 linhas preservadas.
Limitação: a imputação dos nulos será decidida no pipeline, ajustada só no treino. `sexo` sai do modelo antes do merge e foi padronizada apenas para auditoria de fairness.
Para o cliente: "Não informado" em `tipo_renda` (60 linhas) é ausência de dado ou uma categoria própria? As categorias raras (desempregado, licença maternidade, estudante, empresário) devem ser agrupadas?

### `estado_civil`, `ocupacao` e `tipo_organizacao`

Decisão geral: normalizar as strings. Os placeholders "não informado" de `estado_civil` (150), `tipo_organizacao` (10.001, 19,8%) e `tipo_renda` (60) foram convertidos em nulo, como em `sexo`, `possui_carro` e `possui_imovel`. As demais categorias, inclusive as raras, são mantidas. Agrupamentos e tratamento dos nulos ficam para o pipeline, com parâmetros definidos no treino.

`estado_civil`: 5 categorias e 150 nulos (0,3%).

`ocupacao`: 18 categorias e 15.706 nulos (31%). Os nulos não são uniformes: 85,9% dos aposentados (7.964) e os 3 desempregados não têm ocupação, o que sugere ausência com significado. Já assalariados (17,9%), associados comerciais (19,7%), servidores públicos (20,3%) e autônomos (28,8%) também têm nulos, o que sugere dado faltante. O nulo provavelmente mistura os dois casos. Mantido sem imputação nem categoria nova.

`tipo_organizacao`: 57 categorias e 10.001 nulos (19,8%), 44 delas raras (menos de 0,1%). Mantidas como estão.

`tipo_renda`: 9 categorias e 60 nulos (0,1%) após a conversão do placeholder.

Perguntas ao cliente: o nulo de `ocupacao` significa "sem ocupação" para aposentados e desempregados? O que explica nulos em quem tem renda de trabalho? "Não informado" em `estado_civil`, `tipo_organizacao` e `tipo_renda` é ausência ou falha de coleta? Existe hierarquia oficial para agrupar os tipos de organização?

### Decisão: `data_nascimento`

Problema: dois formatos (`AAAA-MM-DD` e `DD/MM/AAAA`) e quatro datas sentinela repetidas de forma anormal: `31/02/1990` (38, data inexistente), `1901-05-20` (38, idade ~124 anos), `2030-01-01` (37, data futura) e `2012-07-15` (38, idade ~12,6 anos). A data real mais frequente aparece 12 vezes.
Regra: converter com `fn.converter_tipos` nos dois formatos; converter as sentinelas em nulo; criar a flag `data_nascimento_invalida`; sem imputação.
Resultado: 151 nulos (0,3%) com flag, 50.600 linhas preservadas.
Limitação: a idade só poderá ser derivada para 50.449 linhas; será calculada com a `data_solicitacao` como referência.
Para o cliente: o que significam essas quatro datas? São valores padrão do sistema de origem?

### `qtd_membros_familia`

Observações:
- 101 registros com valor 0, incoerente porque o solicitante é sempre um membro da família.
- 202 registros com membros < filhos + 1 e 137 com membros < filhos. Hipótese: clientes que não contam filhos que já saíram de casa. É plausível e de impacto pequeno (menos de 0,4% da base).
- 5 registros com mais de 8 membros e 48 com mais de 5 filhos, valores extremos mas possíveis.

Decisão:
- Zeros convertidos em nulo, para imputação no pipeline com parâmetros ajustados só no treino.
- Demais valores mantidos sem alteração.
- Coluna convertida para `Int64`.

É possível que a quantidade de membros menor que a quantidade de filhos +1 se refira a quantos filhos ainda moram com o(a) solicitante. Desta forma, vamos manter
no pipeline apenas a coluna de membros da família.

### `nota_regiao_cliente` e `nota_regiao_cliente_cidade`

- Ordinais de 1 a 3, sem problemas de qualidade, com nota 2 dominante.
- Correlação de Spearman de 0,70, com cerca de 89% dos registros na diagonal.
- Decisão: manter as duas por ora e reavaliar na seleção de variáveis.

### `id_cliente` duplicados

   - **Diagnóstico:** A inspeção comprovou que a segunda ocorrência atua como uma atualização cadastral pontual, corrigindo discrepâncias grosseiras (ex.: clientes com 7 e 8 filhos corrigidos para 2 filhos; preenchimento de datas de nascimento anteriormente nulas). Em contrapartida, a segunda ocorrência introduziu 150 valores ausentes em `estado_civil` que estavam devidamente preenchidos no primeiro registro.
   - **Regra:** Consolidação com prioridade de recência e preenchimento de lacunas (*update with fallback*). Mantém-se o segundo registro como base prioritária e, para quaisquer atributos que resultem em valores ausentes (`NaN`/`NaT`), realiza-se o preenchimento com o valor do primeiro registro (`cad_2a.combine_first(cad_1a)`).
   - **Impacto:** Redução de 50.600 para exatamente 50.000 clientes únicos (chave 1:1), eliminando 600 registros excedentes sem perder dados válidos de histórico.


## Base Submissao

In [ ]:
submissao.info()

In [ ]:
submissao.head()

In [ ]:
resumo, frequencias, variantes = fn.auditar_categoricas(submissao, colunas=["tipo_contrato"])

In [ ]:
resumo.T

In [ ]:
frequencias

In [ ]:
variantes

In [ ]:
# Aplica a padronização só em tipo_contrato.
submissao, rel = fn.padronizar_categoricas(submissao, colunas=["tipo_contrato"])
print(rel.T)

In [ ]:
# Checagens: 3 categorias, mesma contagem de linhas e total por categoria igual ao da auditoria.
print(submissao["tipo_contrato"].value_counts(dropna=False))
print("linhas:", len(submissao))

In [ ]:
# Visualizações gráficas para análise exploratória de dados (EDA), mostrando a frequência de valores e a distribuição 
# de dados numéricos. A função sns.set_theme() aplica um estilo consistente aos gráficos.

sns.set_theme(style="whitegrid")
bruta = submissao_bruta  # leitura apenas; nada é alterado

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.ravel()

# 1. id_cliente: distribuição dos identificadores (busca de faixas ou lacunas).
sns.histplot(bruta["id_cliente"], bins=50, ax=axes[0], color="#4C72B0")
axes[0].set_title("id_cliente: distribuição dos identificadores")

# 2. data_solicitacao: solicitações por dia; datas inválidas viram NaT e ficam fora do gráfico.
datas = pd.to_datetime(bruta["data_solicitacao"], format="mixed", errors="coerce", dayfirst=False)
por_dia = datas.dt.date.value_counts().sort_index()
axes[1].bar(por_dia.index.astype(str), por_dia.values, color="#55A868")
axes[1].set_title(f"data_solicitacao: solicitações por dia ({datas.isna().sum()} datas não convertidas)")
axes[1].tick_params(axis="x", rotation=90)

# 3. dia_semana_solicitacao: contagem por categoria.
sns.countplot(y=bruta["dia_semana_solicitacao"], ax=axes[2],
              order=bruta["dia_semana_solicitacao"].value_counts().index, color="#C44E52")
axes[2].set_title("dia_semana_solicitacao: contagem por categoria")

# 4. hora_solicitacao: hora extraída do texto (horas negativas continuam visíveis).
hora = pd.to_numeric(bruta["hora_solicitacao"].str.split(":").str[0], errors="coerce")
sns.countplot(x=hora, ax=axes[3], color="#8172B2")
axes[3].set_title("hora_solicitacao: contagem por hora (inclui horas negativas)")

# 5. tipo_contrato: contagem por grafia original (mostra as 6 variantes).
sns.countplot(y=bruta["tipo_contrato"], ax=axes[4],
              order=bruta["tipo_contrato"].value_counts().index, color="#CCB974")
axes[4].set_title("tipo_contrato: contagem por grafia original")

# 6 a 8. Valores financeiros em escala log; só valores positivos entram no log.
for ax, col, cor in zip(axes[5:], ["valor_credito", "valor_bem", "valor_parcela"],
                        ["#64B5CD", "#DD8452", "#937860"]):
    serie = bruta[col]
    positivos = serie[serie > 0]
    sns.histplot(positivos, bins=60, log_scale=True, ax=ax, color=cor)
    ax.set_title(f"{col}: escala log | nulos: {serie.isna().sum()} | zeros: {(serie == 0).sum()} | máx: {serie.max():,.0f}")

plt.tight_layout()
plt.show()

In [ ]:
# Visualizações gráficas para análise exploratória de dados (EDA), mostrando a dispersão e os outliers de dados numéricos. 
# A função sns.set_theme() aplica um estilo consistente aos gráficos.

sns.set_theme(style="whitegrid")
bruta = submissao_bruta  # leitura apenas; nada é alterado
colunas = ["valor_credito", "valor_bem", "valor_parcela"]
cores = ["#64B5CD", "#DD8452", "#937860"]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))

for j, (col, cor) in enumerate(zip(colunas, cores)):
    serie = bruta[col].dropna()
    positivos = serie[serie > 0]  # o log só aceita valores positivos

    # Linha de cima: escala linear, mostra o quanto o máximo se afasta da massa.
    sns.boxplot(x=serie, ax=axes[0, j], color=cor, fliersize=3)
    axes[0, j].set_title(f"{col}: escala linear (máx: {serie.max():,.0f})")

    # Linha de baixo: escala log, mostra a dispersão real e os pontos fora do critério de Tukey.
    sns.boxplot(x=positivos, ax=axes[1, j], color=cor, fliersize=3)
    axes[1, j].set_xscale("log")

    # Critério de Tukey calculado no log, onde a variável é aproximadamente simétrica.
    log_vals = np.log10(positivos)
    q1, q3 = log_vals.quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_sup = 10 ** (q3 + 1.5 * iqr)
    limite_inf = 10 ** (q1 - 1.5 * iqr)
    n_acima = (positivos > limite_sup).sum()
    n_abaixo = (positivos < limite_inf).sum()
    axes[1, j].set_title(f"{col}: escala log | fora de Tukey: {n_acima} acima, {n_abaixo} abaixo")

plt.tight_layout()
plt.show()

### Visão geral das variáveis originais (base bruta)

Os gráficos abaixo usam a base bruta, antes de qualquer tratamento, para que cada decisão tenha uma evidência visual de origem. Foram usados histogramas e contagens para as variáveis de frequência, e boxplots (escala linear e log) para as variáveis financeiras. O critério de Tukey (1,5 vez o intervalo interquartil) foi calculado no log, onde as variáveis são aproximadamente simétricas.

| Variável | O que o gráfico mostra | Decisão que sustenta |
|---|---|---|
| `id_cliente` | Densidade uniforme (cerca de 800 por faixa) até perto de 460.000, com degrau no final (3.200 e 7.400 por faixa) | A investigar, pois conversa com os 400 clientes repetidos; entra na decisão de deduplicação |
| `data_solicitacao` | Volume diário estável (cerca de 2.700), com quedas nos fins de semana (dias 08, 09, 15, 16, 22 e 23); 51 datas não convertidas | Datas válidas coerentes; as 51 inválidas (`2025-13-01`) viram nulo, com flag `data_invalida` |
| `dia_semana_solicitacao` | Variantes de grafia (`segunda feira`, `sexta feira `, `TERÇA-FEIRA`) além das categorias padrão | Remover a coluna original e derivar `dia_semana` da data |
| `hora_solicitacao` | Distribuição concentrada entre 8h e 17h, com valores -1 e 25, fora do intervalo de 0 a 23, e vazio em 22 | Sem tratamento por decisão do projeto; registrado como pergunta ao cliente |
| `tipo_contrato` | 6 grafias para 3 categorias reais; as variantes minoritárias são pouco visíveis | Padronização por normalização, sem perda de linhas |
| `valor_parcela` | Distribuição em log aproximadamente em sino; 67 zeros e 5 nulos; cauda contínua, mínimo positivo de 789 | Zeros convertidos em nulo; sem correção de outliers |
| `valor_bem` | 2.460 nulos; picos em valores redondos (perto de 2,5×10⁵, 4,5×10⁵, 7×10⁵ e 10⁶); cauda contínua até 4.050.000 | Manter nulo, sem imputação; flag `valor_bem_ausente`; picos registrados como característica da variável |
| `valor_credito` | Na escala linear, a caixa fica colada ao zero e há pontos isolados de 10 a 135 milhões; no log, um vazio separa o bloco principal (até ~3 milhões) de um grupo de 10 a 135 milhões | Correção de escala (÷100) nas 63 linhas com razão crédito/bem entre 80 e 200, flag `credito_corrigido` |

#### Boxplots das variáveis financeiras

| Variável | Máximo | Fora de Tukey (log) | Leitura |
|---|---|---|---|
| `valor_credito` | 135.000.000 | 67 acima, 558 abaixo | Grupo separado por um vazio da massa principal |
| `valor_bem` | 4.050.000 | 1 acima, 65 abaixo | Cauda contínua e gradual, sem bloco separado |
| `valor_parcela` | 302.580 | 79 acima, 987 abaixo | Cauda longa e contínua, sem vazio |

Leitura principal: o que distingue `valor_credito` das outras duas variáveis não é a existência de pontos fora do critério, que todas têm, mas a descontinuidade. Em `valor_bem` e `valor_parcela` os pontos extremos formam uma cauda contínua, compatível com a assimetria natural de valores financeiros. Em `valor_credito` há um grupo isolado, o que reforça a hipótese de erro de escala sustentada pela razão crédito/bem e pela comparação com o histórico.

Observações e cuidados:

- Os 67 pontos acima de Tukey em `valor_credito` são próximos das 63 linhas corrigidas, mas não iguais. O limite cai um pouco abaixo do bloco inflado e inclui algumas linhas legítimas de valor alto, então Tukey não substitui a regra da razão com o bem.
- Os pontos abaixo do limite inferior (558, 65 e 987) correspondem à cauda de valores pequenos e não indicam erro por si só; nenhum foi tratado.
- Tukey é um critério estatístico e não prova que o valor está errado; a correção se apoia na combinação de evidências (razão com o bem, parcela normal e ausência do padrão no histórico).
- A ausência de uma segunda corcova visível no histograma de `valor_credito` se explica pelo tamanho do grupo: 63 linhas em 50.400 ficam invisíveis na escala de contagem. O boxplot em escala linear torna esse grupo evidente.

In [ ]:
# Quantos nulos e zeros existem em valor_parcela.
print("nulos:", submissao["valor_parcela"].isna().sum())
print("zeros:", (submissao["valor_parcela"] == 0).sum())
print("negativos:", (submissao["valor_parcela"] < 0).sum())

# Distribuição das parcelas positivas, para saber o que é "normal".
print(submissao.loc[submissao["valor_parcela"] > 0, "valor_parcela"].describe())

# As linhas com zero: o crédito delas é positivo? Em que tipos de contrato aparecem?
zeros = submissao[submissao["valor_parcela"] == 0]
print("\ncrédito nas linhas com parcela zero:")
print(zeros["valor_credito"].describe())
print(zeros["tipo_contrato"].value_counts())

# Os 5 nulos, para inspeção direta.
print(submissao[submissao["valor_parcela"].isna()])

In [ ]:
nulos_antes = submissao["valor_parcela"].isna().sum()

# Parcela zero com crédito positivo é inconsistente: trocamos por nulo.
# .replace(0, np.nan) só afeta valores exatamente iguais a zero.
submissao["valor_parcela"] = submissao["valor_parcela"].replace(0, np.nan)

# Checagens: esperamos 5 -> 72 nulos, 0 zeros e a mesma quantidade de linhas.
print("nulos antes:", nulos_antes, "| depois:", submissao["valor_parcela"].isna().sum())
print("zeros restantes:", (submissao["valor_parcela"] == 0).sum())
print("linhas:", len(submissao))

In [ ]:
# Nulos, zeros e negativos em valor_bem.
print("nulos:", submissao["valor_bem"].isna().sum())
print("zeros:", (submissao["valor_bem"] == 0).sum())
print("negativos:", (submissao["valor_bem"] < 0).sum())

# Distribuição dos valores preenchidos, para saber o que é "normal".
print(submissao["valor_bem"].describe())

# Coluna auxiliar: 1 quando valor_bem é nulo, 0 caso contrário.
# A média dessa coluna dentro de cada grupo é exatamente a proporção de nulos.
bem_nulo = np.where(submissao["valor_bem"].isna(), 1, 0)

# Taxa de nulos por tipo de contrato: se for parecida, a ausência não depende do produto.
print(pd.Series(bem_nulo, index=submissao.index).groupby(submissao["tipo_contrato"]).mean())

# Os nulos têm crédito e parcela diferentes dos demais? Compara as medianas.
print(submissao.groupby(submissao["valor_bem"].isna())[["valor_credito", "valor_parcela"]].median())

In [ ]:
# Flag de ausência: 1 quando valor_bem é nulo, 0 caso contrário.
submissao["valor_bem_ausente"] = np.where(submissao["valor_bem"].isna(), 1, 0)

# Distribuição do crédito em cada grupo: se os intervalos mal se sobrepõem,
# existe uma regra de negócio por trás da ausência.
print(submissao.groupby("valor_bem_ausente")["valor_credito"].describe())

# Checagens: 2.460 flags iguais a 1 e nenhuma linha perdida.
print("flag = 1:", submissao["valor_bem_ausente"].sum())
print("linhas:", len(submissao))

In [ ]:
# Razão crédito/bem, calculada só onde o bem existe e é positivo.
razao = np.where(submissao["valor_bem"] > 0, submissao["valor_credito"] / submissao["valor_bem"], np.nan)
submissao["razao_credito_bem"] = razao

# Distribuição da razão: o esperado é concentrar perto de 1, com uma cauda isolada em ~100.
print(submissao["razao_credito_bem"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99, 0.999]))

# Quantas linhas caem em cada faixa.
faixas = pd.cut(submissao["razao_credito_bem"], bins=[0, 0.5, 1.5, 5, 80, 200, np.inf])
print(faixas.value_counts(sort=False))

# As maiores razões, com os valores originais, para inspeção direta.
print(submissao.nlargest(10, "valor_credito")[["id_cliente", "tipo_contrato", "valor_credito", "valor_bem", "valor_parcela", "razao_credito_bem"]])

In [ ]:
print(list(emprestimos.columns))
# Seleciona os 63 contratos suspeitos pela mesma regra de razão (80 a 200), sem alterar nada.
suspeitos = submissao[(submissao["razao_credito_bem"] > 80) & (submissao["razao_credito_bem"] < 200)]
print("contratos suspeitos:", len(suspeitos), "| clientes distintos:", suspeitos["id_cliente"].nunique())

In [ ]:
# Passo 1: quantos dos 63 clientes suspeitos aparecem no histórico?
ids_suspeitos = suspeitos["id_cliente"].unique()
hist_susp = emprestimos[emprestimos["id_cliente"].isin(ids_suspeitos)]
print("clientes suspeitos com histórico:", hist_susp["id_cliente"].nunique(), "de", len(ids_suspeitos))
print("contratos desses clientes no histórico:", len(hist_susp))

# Passo 2: o histórico tem o mesmo problema de escala?
# Razão crédito/bem calculada só onde bem e crédito existem e são positivos.
ok = (emprestimos["valor_bem"] > 0) & (emprestimos["valor_credito"] > 0)
emprestimos_razao = np.where(ok, emprestimos["valor_credito"] / emprestimos["valor_bem"], np.nan)
emprestimos["razao_credito_bem"] = emprestimos_razao
print(emprestimos["razao_credito_bem"].describe(percentiles=[0.01, 0.5, 0.99, 0.999]))
print("histórico com razão > 80:", (emprestimos["razao_credito_bem"] > 80).sum())

# Passo 3: para cada cliente suspeito, compara o crédito suspeito com a mediana dos créditos passados.
mediana_hist = (
    hist_susp[hist_susp["valor_credito"] > 0]
    .groupby("id_cliente")["valor_credito"].median()
    .rename("mediana_credito_hist")
)
comparacao = suspeitos[["id_cliente", "valor_credito", "valor_bem"]].merge(
    mediana_hist, on="id_cliente", how="inner"
)
comparacao["credito_sobre_hist"] = comparacao["valor_credito"] / comparacao["mediana_credito_hist"]
print(len(comparacao), "clientes com crédito histórico positivo")
print(comparacao["credito_sobre_hist"].describe())
print(comparacao.sort_values("credito_sobre_hist").head(10))

In [ ]:
# Guarda o valor original antes de qualquer alteração, para auditoria.
submissao["valor_credito_original"] = submissao["valor_credito"]

# Flag: 1 onde a razão crédito/bem está na faixa de erro de escala (80 a 200).
# Linhas sem bem têm razão NaN; a comparação com NaN dá False, então ficam 0.
erro_escala = (submissao["razao_credito_bem"] > 80) & (submissao["razao_credito_bem"] < 200)
submissao["credito_corrigido"] = np.where(erro_escala, 1, 0)

# Corrige dividindo por 100 só onde a flag é 1.
submissao["valor_credito"] = np.where(
    submissao["credito_corrigido"] == 1,
    submissao["valor_credito"] / 100,
    submissao["valor_credito"],
)

# Recalcula a razão com o crédito corrigido.
submissao["razao_credito_bem"] = np.where(
    submissao["valor_bem"] > 0, submissao["valor_credito"] / submissao["valor_bem"], np.nan
)

# Checagens
print("corrigidas:", submissao["credito_corrigido"].sum())
print(submissao["razao_credito_bem"].describe(percentiles=[0.01, 0.99, 0.999]))
print("máximo de valor_credito:", submissao["valor_credito"].max())
print("linhas:", len(submissao))

In [ ]:
# Teste: converte só a data, guardando o resultado em outro DataFrame para não alterar submissao.
teste, rel_datas = fn.converter_tipos(
    submissao, datas={"data_solicitacao": ["%Y-%m-%d", "%d/%m/%Y"]}
)
print(rel_datas.T)

# Período das datas convertidas: esperamos 2025-02-04 a 2025-02-24.
print(teste["data_solicitacao"].min(), teste["data_solicitacao"].max())

In [ ]:
# Verifica os dias da semana registrados nas 51 linhas com '2025-13-01'.
mascara_1301 = submissao_bruta["data_solicitacao"].astype(str).str.strip().eq("2025-13-01")
display(submissao_bruta.loc[mascara_1301, "dia_semana_solicitacao"].value_counts(dropna=False))

In [ ]:
# Só converte se a coluna ainda for texto (seguro para reexecução).
if not pd.api.types.is_datetime64_any_dtype(submissao["data_solicitacao"]):
    submissao, rel_datas = fn.converter_tipos(
        submissao, datas={"data_solicitacao": ["%Y-%m-%d", "%d/%m/%Y"]}
    )

# Flag derivada do bruto: 1 onde a data original era o placeholder '2025-13-01'.
submissao["data_invalida"] = np.where(
    submissao_bruta["data_solicitacao"] == "2025-13-01", 1, 0
)

# Checagens: 51 nulos, 51 flags, as duas coincidem e 50.400 linhas.
print("nulos:", submissao["data_solicitacao"].isna().sum())
print("flag = 1:", submissao["data_invalida"].sum())
print("flag coincide com nulo:", (submissao["data_invalida"] == submissao["data_solicitacao"].isna().astype(int)).all())
print("período:", submissao["data_solicitacao"].min(), "->", submissao["data_solicitacao"].max())
print("linhas:", len(submissao))

In [ ]:
nomes = {0: "segunda-feira", 1: "terça-feira", 2: "quarta-feira", 3: "quinta-feira",
         4: "sexta-feira", 5: "sábado", 6: "domingo"}

derivado = submissao["data_solicitacao"].dt.dayofweek.map(nomes)
original = submissao["dia_semana_solicitacao"].str.lower().str.strip()

valida = submissao["data_solicitacao"].notna()
concorda = np.where(derivado == original, 1, 0)

print("válidas:", valida.sum())
print("concordam:", concorda[valida].sum(), "| discordam:", (1 - concorda[valida]).sum())
print(pd.crosstab(derivado[valida], original[valida]))
print(derivado[valida].value_counts())

In [ ]:
submissao.info()

In [ ]:
# Separa a 1ª e a 2ª ocorrência dos IDs repetidos em submissao preservando a ordem original da tabela.
dup_sub_1 = submissao.loc[submissao.duplicated("id_cliente", keep="last")].set_index("id_cliente").sort_index()
dup_sub_2 = submissao.loc[submissao.duplicated("id_cliente", keep="first")].set_index("id_cliente").sort_index()

# Contabiliza em quantos pares cada coluna diverge entre a 1ª e a 2ª ocorrência.
dif_sub = (dup_sub_1.ne(dup_sub_2) & ~(dup_sub_1.isna() & dup_sub_2.isna())).sum()

print(f"Clientes repetidos: {len(dup_sub_1)} ({len(dup_sub_1) * 2} linhas)")
print(f"2ª ocorrência está 100% concentrada em index >= 50000? {np.array_equal(submissao.index >= 50000, submissao.duplicated('id_cliente', keep='first'))}")
print(f"Crédito com decimais quebrados -> 1ª ocorrência: {(dup_sub_1['valor_credito'] % 0.5 != 0).sum()} | 2ª ocorrência: {(dup_sub_2['valor_credito'] % 0.5 != 0).sum()}")
print("\nColunas que divergem entre a 1ª e a 2ª ocorrência:")
display(dif_sub.loc[dif_sub > 0])

## Base de histórico de empréstimos

In [ ]:
print(emprestimos.shape)
emprestimos.info()

nulos = emprestimos.isna().sum()
print("nulos por coluna:")
print(nulos[nulos > 0].sort_values(ascending=False))

print("linhas duplicadas:", emprestimos.duplicated().sum())
print("id_contrato únicos:", emprestimos["id_contrato"].nunique())
print("id_cliente únicos:", emprestimos["id_cliente"].nunique())

In [ ]:
emprestimos.head()

In [ ]:
# Análise da relação entre status_contrato com valores ausentes no dataframe
cols_nulo = ["data_liberacao", "data_primeiro_vencimento", "data_ultimo_vencimento_original",
             "data_ultimo_vencimento", "data_encerramento", "valor_parcela",
             "qtd_parcelas_planejadas", "valor_entrada", "taxa_juros_padrao",
             "indicador_seguro_contratado", "acompanhantes_cliente"]
print(emprestimos[cols_nulo].isna().groupby(emprestimos["status_contrato"]).mean().round(3).T)
print(emprestimos["status_contrato"].value_counts(dropna=False))
print(emprestimos["tipo_contrato"].value_counts(dropna=False))

In [ ]:
# Define as colunas de texto do histórico que serão auditadas.
colunas_texto = ["tipo_contrato", "status_contrato", "tipo_pagamento",
                 "finalidade_emprestimo", "tipo_cliente", "faixa_rendimento",
                 "tipo_portfolio", "tipo_produto", "categoria_bem",
                 "combinacao_produto", "setor_vendedor", "canal_venda",
                 "dia_semana_solicitacao", "indicador_ultima_solicitacao_contrato",
                 "motivo_recusa", "acompanhantes_cliente"]

resumo, freq, variantes = fn.auditar_categoricas(emprestimos, colunas_texto)
print(resumo)

# Lista as colunas de data que ainda estão como texto.
colunas_data = ["data_decisao", "data_primeiro_vencimento", "data_liberacao",
                "data_ultimo_vencimento_original", "data_ultimo_vencimento",
                "data_encerramento"]

# Mostra os comprimentos das strings (revelam formatos mistos) e 3 exemplos de cada coluna de data.
for col in colunas_data:
    print(col, emprestimos[col].dropna().str.len().value_counts().to_dict())
    print(emprestimos[col].dropna().sample(3, random_state=1).tolist())

# Inspeciona o formato de hora_solicitacao, que está como texto.
print(emprestimos["hora_solicitacao"].sample(5, random_state=1).tolist())

# Confere quantos valores distintos têm area_venda e indicador_ultima_solicitacao_dia (provável código, não quantidade).
print(emprestimos[["area_venda", "indicador_ultima_solicitacao_dia"]].nunique())

In [ ]:
# Define as colunas categóricas que serão normalizadas.
colunas_categoricas_emp = [
    "tipo_contrato",
    "status_contrato",
    "tipo_pagamento",
    "finalidade_emprestimo",
    "tipo_cliente",
    "faixa_rendimento",
    "tipo_portfolio",
    "tipo_produto",
    "categoria_bem",
    "combinacao_produto",
    "setor_vendedor",
    "canal_venda",
    "dia_semana_solicitacao",
    "indicador_ultima_solicitacao_contrato",
    "motivo_recusa",
    "acompanhantes_cliente",
]

# Padroniza as categóricas e converte os placeholders definidos na função em nulos.
emprestimos, rel_categoricas_emp = fn.padronizar_categoricas(
    emprestimos,
    colunas_categoricas_emp,
)

# Exibe o relatório de alterações nas colunas categóricas.
print(rel_categoricas_emp)

# Define as colunas de data que serão convertidas para datetime.
colunas_data_emp = [
    "data_decisao",
    "data_liberacao",
    "data_primeiro_vencimento",
    "data_ultimo_vencimento_original",
    "data_ultimo_vencimento",
    "data_encerramento",
]

# Define e associa o formato observado nas strings de data.
formato_data_emp = "%Y-%m-%d"
formatos_datas_emp = {
    coluna: formato_data_emp
    for coluna in colunas_data_emp
}

# Converte as datas e registra valores que não puderem ser interpretados.
emprestimos, rel_datas_emp = fn.converter_tipos(
    emprestimos,
    datas=formatos_datas_emp,
)

# Exibe o relatório de conversão das datas, incluindo eventuais falhas.
print(rel_datas_emp)

### Padronização de categóricas e conversão de datas — `emprestimos`

A análise foi realizada sobre uma cópia de trabalho de `emprestimos_bruta`, mantendo a base bruta intacta. A base contém 211.944 linhas e 37 colunas; não há linhas totalmente duplicadas, `id_contrato` tem 211.944 valores únicos e `id_cliente`, 47.952.

As 16 colunas categóricas auditadas foram padronizadas com `padronizar_categoricas`: textos convertidos para minúsculas, sem acentos e com espaços normalizados. Os placeholders definidos pela função foram convertidos em nulos. `tipo_contrato` passou de 6 para 3 categorias, consolidando variantes de grafia. Também foram convertidos em nulos placeholders de `tipo_pagamento` (79.757), `finalidade_emprestimo` (85.952), `tipo_cliente` (252), `faixa_rendimento` (65.804), `tipo_portfolio` (35.485), `tipo_produto` (135.339), `categoria_bem` (98.847), `setor_vendedor` (108.594) e `motivo_recusa` (522). `combinacao_produto` manteve seus 27 nulos originais, e `acompanhantes_cliente`, seus 104.383 nulos originais. Nenhuma linha foi removida.

As colunas `data_decisao`, `data_liberacao`, `data_primeiro_vencimento`, `data_ultimo_vencimento_original`, `data_ultimo_vencimento` e `data_encerramento` foram convertidas de texto para `datetime`, usando o formato `%Y-%m-%d`. A conversão não gerou falhas nem nulos adicionais.

In [ ]:
# Define as colunas numéricas de interesse, excluindo identificadores e códigos.
colunas_numericas_emp = [
    "valor_solicitado",
    "valor_credito",
    "valor_bem",
    "valor_parcela",
    "valor_entrada",
    "percentual_entrada",
    "qtd_parcelas_planejadas",
    "taxa_juros_padrao",
    "taxa_juros_promocional",
    "indicador_seguro_contratado",
    "razao_credito_bem",
]

# Mantém apenas as colunas numéricas que existem na base de trabalho.
colunas_numericas_emp = [
    coluna for coluna in colunas_numericas_emp
    if coluna in emprestimos.columns
]

# Exibe estatísticas descritivas, incluindo os nulos.
print(emprestimos[colunas_numericas_emp].describe().T)

# Conta valores negativos e zeros em cada coluna numérica, sem modificar a base.
for coluna in colunas_numericas_emp:
    serie = emprestimos[coluna].dropna()
    print(
        f"{coluna}: negativos={(serie < 0).sum()}, "
        f"zeros={(serie == 0).sum()}, nulos={emprestimos[coluna].isna().sum()}"
    )


In [ ]:
# Define pares de datas cuja ordem cronológica vale a pena conferir.
pares_datas_emp = [
    ("data_decisao", "data_liberacao"),
    ("data_decisao", "data_primeiro_vencimento"),
    ("data_primeiro_vencimento", "data_ultimo_vencimento_original"),
    ("data_ultimo_vencimento_original", "data_ultimo_vencimento"),
    ("data_ultimo_vencimento", "data_encerramento"),
]

# Conta casos em que a data posterior é anterior à data inicial do par.
for data_inicial, data_final in pares_datas_emp:
    datas_preenchidas = emprestimos[data_inicial].notna() & emprestimos[data_final].notna()
    inversoes = (
        datas_preenchidas
        & (emprestimos[data_final] < emprestimos[data_inicial])
    ).sum()
    print(
        f"{data_inicial} > {data_final}: {inversoes} casos "
        f"entre {datas_preenchidas.sum()} pares preenchidos"
    )

In [ ]:
# Seleciona contratos com data de liberação anterior à data de decisão.
mascara_liberacao = (
    emprestimos["data_decisao"].notna()
    & emprestimos["data_liberacao"].notna()
    & (emprestimos["data_liberacao"] < emprestimos["data_decisao"])
)

# Exibe todos os registros encontrados, com todas as colunas da base.
print(f"Data de liberação anterior à decisão: {mascara_liberacao.sum()} casos")
display(emprestimos.loc[mascara_liberacao])


# Seleciona contratos com primeiro vencimento anterior à data de decisão.
mascara_primeiro_vencimento = (
    emprestimos["data_decisao"].notna()
    & emprestimos["data_primeiro_vencimento"].notna()
    & (emprestimos["data_primeiro_vencimento"] < emprestimos["data_decisao"])
)

# Exibe até 20 registros encontrados, com todas as colunas da base.
print(
    "Primeiro vencimento anterior à decisão: "
    f"{mascara_primeiro_vencimento.sum()} casos; exibindo até 20 exemplos"
)
display(emprestimos.loc[mascara_primeiro_vencimento].head(20))

### Inconsistências cronológicas pendentes de validação

Na análise das datas dos contratos, foram identificados 6 registros em que data_liberacao antecede data_decisao. Todos têm status aprovado. A diferença entre as datas varia de 243 a 1.261 dias, com mediana de 630 dias.

Também foram encontrados 198 registros em que data_primeiro_vencimento antecede data_decisao. Todos têm status aprovado. A diferença varia de 4 a 2.809 dias, com mediana de 898 dias. Os casos distribuem-se entre financiamento de consumo (86), emprestimo em dinheiro (75) e credito rotativo (37).

Embora essas relações cronológicas pareçam atípicas, não é possível concluir, com as informações disponíveis, que as datas estejam incorretas. Há uma base adicional que identifica contratos renegociados; sua integração poderá fornecer contexto e ajudar a explicar parte dos casos.

Por isso, os registros e as datas originais foram mantidos sem correção ou exclusão. A análise será retomada após a integração das bases, considerando o histórico de renegociação.

In [ ]:
emprestimos.info()

In [ ]:
# Resume a distribuição da razão crédito/bem, ignorando valores ausentes.
print("Valores ausentes:", emprestimos["razao_credito_bem"].isna().sum())
display(
    emprestimos["razao_credito_bem"]
    .describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99])
)

# Quantifica valores não positivos e valores acima de 1.
razao = emprestimos["razao_credito_bem"]

print("Valores menores ou iguais a zero:", razao.le(0).sum())
print("Valores acima de 1:", razao.gt(1).sum())
print("Valores acima de 2:", razao.gt(2).sum())

In [ ]:
# Inspeciona os registros com razão crédito/bem acima de 2.
mascara_razao_alta = emprestimos["razao_credito_bem"] > 2

print(f"Casos com razão acima de 2: {mascara_razao_alta.sum()}")
display(emprestimos.loc[mascara_razao_alta])

In [ ]:
emprestimos['status_contrato'].value_counts(dropna=False)

In [ ]:
# Define recusa como 1 e os demais status como 0.
recusa = emprestimos["status_contrato"].eq("recusado").astype("int8")

print(f"Percentual de recusados: {recusa.mean():.2%}")

# Calcula correlações entre variáveis numéricas e o indicador de recusa.
colunas_numericas = emprestimos.select_dtypes(include="number").columns.difference(
    ["id_contrato", "id_cliente"]
)

correlacoes = (
    emprestimos[colunas_numericas]
    .corrwith(recusa)
    .dropna()
    .sort_values(key=lambda serie: serie.abs(), ascending=False)
    .head(15)
    .sort_values()
    .rename_axis("variavel")
    .reset_index(name="correlacao")
)

# Mostra as variáveis numéricas com maior associação com a recusa.
plt.figure(figsize=(8, 5))
sns.barplot(data=correlacoes, x="correlacao", y="variavel")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Associação entre variáveis numéricas e recusa")
plt.xlabel("Correlação com indicador de recusa")
plt.ylabel("")
plt.tight_layout()
plt.show()

# Seleciona variáveis categóricas, excluindo identificadores, alvo e motivo da recusa.
colunas_categoricas = emprestimos.select_dtypes(
    include=["object", "str", "category"]
).columns.difference(
    ["status_contrato", "motivo_recusa"]
)

for coluna in colunas_categoricas:
    # Calcula a taxa de recusa e o número de registros por categoria.
    resumo = (
        pd.DataFrame({
            "categoria": emprestimos[coluna].astype("string").fillna("ausente"),
            "recusa": recusa,
        })
        .groupby("categoria", observed=True)["recusa"]
        .agg(taxa_recusa="mean", quantidade="size")
        .reset_index()
    )

    # Exibe somente categorias com pelo menos 100 registros e recusa observada.
    resumo = resumo.loc[
        (resumo["quantidade"] >= 100) & (resumo["taxa_recusa"] > 0)
    ]

    # Não cria gráfico para variáveis sem categorias que atendam aos filtros.
    if resumo.empty:
        continue

    plt.figure(figsize=(8, max(3, 0.3 * len(resumo))))
    sns.barplot(data=resumo, x="taxa_recusa", y="categoria")
    plt.title(f"Taxa de recusa por categoria — {coluna}")
    plt.xlabel("Taxa de recusa")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

In [ ]:
emprestimos['motivo_recusa'].value_counts(dropna=False)

In [ ]:
# Seleciona os valores monetários para análise por status.
colunas_valores = [
    "valor_solicitado",
    "valor_credito",
    "valor_bem",
    "valor_parcela",
    "valor_entrada",
]

# Resume preenchimento e distribuição dos valores dentro de cada status.
display(
    emprestimos.groupby("status_contrato", observed=True)[colunas_valores]
    .describe(percentiles=[0.50, 0.95, 0.99])
    .T
)

# Conta ausências, zeros e negativos por variável e status, sem modificar a base.
for coluna in colunas_valores:
    indicadores = pd.DataFrame({
        "status_contrato": emprestimos["status_contrato"],
        "ausentes": emprestimos[coluna].isna(),
        "zeros": emprestimos[coluna].eq(0),
        "negativos": emprestimos[coluna].lt(0),
    })

    print(f"\n{coluna}")
    display(
        indicadores.groupby("status_contrato", observed=True)
        [["ausentes", "zeros", "negativos"]]
        .sum()
    )

### Razão crédito/bem e análise de recusas

- `razao_credito_bem`: 161.422 valores preenchidos e 50.522 ausentes.
- Mediana: 1,00; percentil 99: 1,36; máximo: 6,67.
- Não há valores não positivos.
- Dos 30 contratos com razão acima de 2, 29 foram recusados e 1 aprovado.
- Nenhum valor foi corrigido ou excluído; valores extremos não foram considerados erros confirmados.

Contratos recusados representam 17,15% da base. Foram exploradas correlações ponto-bisseriais para variáveis numéricas e taxas de recusa por categoria.

Essas associações não implicam causalidade nem comprovam importância para prever inadimplência. Não foi aplicado scaler, pois a padronização linear não altera a correlação.

### Decisão sobre histórico de recusas

A criação dessas features foi adiada: recusas anteriores podem refletir políticas antigas ou condições financeiras que já mudaram.

O `motivo_recusa` do contrato atual não será utilizado como feature quando indisponível no momento da previsão, evitando vazamento. Nenhum contrato foi removido por recusa.


In [ ]:
# Aplica o estilo e seleciona as variáveis numéricas, exceto identificadores.
sns.set_theme(style="whitegrid")

colunas_numericas = emprestimos.select_dtypes(include="number").columns.difference(
    ["id_cliente", "id_contrato"],
    sort=False,
)

# Divide os gráficos em figuras menores para facilitar leitura e renderização.
for inicio in range(0, len(colunas_numericas), 4):
    colunas = colunas_numericas[inicio:inicio + 4]

    fig, axes = plt.subplots(
        len(colunas), 2,
        figsize=(15, 3.5 * len(colunas)),
        squeeze=False,
    )

    for linha, coluna in enumerate(colunas):
        # Remove apenas os nulos da visualização; a base permanece intacta.
        serie = emprestimos[coluna]
        valores = serie.dropna().to_numpy()

        if valores.size == 0:
            for ax in axes[linha]:
                ax.set_title(f"{coluna}: sem valores preenchidos")
                ax.set_axis_off()
            continue

        # Mostra a distribuição completa, incluindo zeros e valores negativos.
        axes[linha, 0].hist(valores, bins=50, color="#4C72B0")
        axes[linha, 0].set_title(
            f"{coluna} | nulos: {serie.isna().sum():,} "
            f"| zeros: {serie.eq(0).sum():,}"
        )
        axes[linha, 0].set_ylabel("Frequência")
        axes[linha, 0].set_xlabel(coluna)

        # Mantém os pontos extremos visíveis no boxplot.
        axes[linha, 1].boxplot(
            valores,
            vert=False,
            flierprops={
                "marker": ".",
                "markersize": 2,
                "alpha": 0.3,
            },
        )
        axes[linha, 1].set_title(f"{coluna}: boxplot")
        axes[linha, 1].set_xlabel(coluna)
        axes[linha, 1].set_yticks([])

    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
# 1. Seleciona colunas textuais/categóricas de submissao (exceto datas/horários)
cols_cat_sub = [
    c for c in submissao.select_dtypes(include=["object", "string", "category"]).columns 
    if c not in ["data_solicitacao", "hora_solicitacao"]
]

# 2. Aplica padronização em cópia isolada de submissao
sub_teste, rel_sub = fn.padronizar_categoricas(submissao, colunas=cols_cat_sub)

# 3. Separa 1ª e 2ª ocorrências alinhadas por id_cliente
sub_teste_1 = sub_teste.loc[sub_teste.duplicated("id_cliente", keep="last")].set_index("id_cliente").sort_index()
sub_teste_2 = sub_teste.loc[sub_teste.duplicated("id_cliente", keep="first")].set_index("id_cliente").sort_index()

# 4. Inspeciona os 6 casos divergentes de tipo_contrato
mascara_tc = sub_teste_1["tipo_contrato"] != sub_teste_2["tipo_contrato"]
print("--- Casos com divergência em tipo_contrato pós-padronização ---")
display(pd.DataFrame({
    "tipo_1a (<50k)": sub_teste_1.loc[mascara_tc, "tipo_contrato"],
    "tipo_2a (>=50k)": sub_teste_2.loc[mascara_tc, "tipo_contrato"]
}))

# 5. Inspeciona casos com divergência em datas e valores de bem
cols_outras = ["data_solicitacao", "dia_semana_solicitacao", "valor_bem"]
mascara_outras = (sub_teste_1[cols_outras].ne(sub_teste_2[cols_outras]) & ~(sub_teste_1[cols_outras].isna() & sub_teste_2[cols_outras].isna())).any(axis=1)

print("\n--- Casos com divergência em data ou valor do bem em submissao ---")
display(pd.concat([
    sub_teste_1.loc[mascara_outras, cols_outras].add_suffix("_1a"),
    sub_teste_2.loc[mascara_outras, cols_outras].add_suffix("_2a")
], axis=1))

### Distribuição das variáveis numéricas — empréstimos

### Valores monetários
- `valor_solicitado`, `valor_credito`, `valor_bem`, `valor_parcela` e `valor_entrada` apresentam forte assimetria à direita, com muitos extremos pelo critério do boxplot.
- `valor_solicitado` e `valor_credito` não possuem nulos, mas apresentam 44.179 e 37.968 zeros, respectivamente. O significado desses zeros depende do contexto operacional e do status.
- `valor_bem` possui 49.784 ausências; `valor_parcela`, 51.265; e `valor_entrada`, 113.618.
- As ausências variam por status, conforme a análise anterior; não serão imputadas indiscriminadamente.

### Entrada e prazo
- `percentual_entrada` concentra registros em zero e próximo de 0,10. Possui a mesma quantidade de nulos e zeros de `valor_entrada`, sem que isso confirme correspondência registro a registro.
- `qtd_parcelas_planejadas` apresenta picos em prazos específicos, compatíveis com uma variável discreta. Há 51.264 nulos e 16.183 zeros, cujo significado permanece pendente de validação.

### Taxas e área de venda
- As duas taxas de juros possuem 211.233 ausências cada, aproximadamente 99,66% da base. Permanecem candidatas à exclusão, sem decisão definitiva.
- As taxas apresentam concentrações distintas; sua escala e interpretação operacional precisam ser confirmadas.
- `area_venda` apresenta forte assimetria e valores extremos próximos de 120 mil. Unidade e significado devem ser validados antes de qualquer tratamento.

### Indicadores e razão crédito/bem
- `indicador_ultima_solicitacao_dia` é quase constante: apenas 706 registros são zero. Sua disponibilidade no momento da previsão deve ser verificada para evitar informação futura.
- `indicador_seguro_contratado` possui 85.165 ausências. Ausência não será interpretada automaticamente como ausência de seguro.
- Boxplots de indicadores binários não são adequados para identificar erros ou outliers.
- `razao_credito_bem` concentra-se próximo de 1, com poucos valores muito elevados, conforme a análise já documentada.

### Decisão
Nenhum valor numérico foi alterado, imputado ou excluído nesta etapa. Os extremos estatísticos não foram classificados como erros confirmados. A análise seguirá para a base `parcelas`.

## Base Parcelas

In [ ]:
parcelas.info()

In [ ]:
parcelas.head()

In [ ]:
# Converte as datas na cópia de trabalho, preservando parcelas_bruta.
parcelas, relatorio_datas = fn.converter_tipos(
    parcelas,
    datas={
        "data_prevista_pagamento": "%Y-%m-%d",
        "data_real_pagamento": "%Y-%m-%d",
    },
)

display(relatorio_datas)

In [ ]:
# Examina as versões existentes sem alterar seu tipo.
display(
    parcelas["versao_parcela"]
    .value_counts(dropna=False)
    .sort_index()
    .rename("quantidade")
    .to_frame()
)

# Conta duplicatas integrais e registros envolvidos em repetição da chave candidata.
chave_parcela = ["id_contrato", "versao_parcela", "numero_parcela"]

print(f"Duplicatas integrais: {parcelas.duplicated().sum():,}")
print(
    "Registros com chave candidata repetida: "
    f"{parcelas.duplicated(chave_parcela, keep=False).sum():,}"
)

# Verifica se cada contrato está associado a apenas um cliente.
clientes_por_contrato = (
    parcelas.groupby("id_contrato", sort=False)["id_cliente"].nunique()
)

print(
    "Contratos associados a mais de um cliente: "
    f"{clientes_por_contrato.gt(1).sum():,}"
)

# Compara ausência de data e valor pago, sem criar colunas na base.
display(
    pd.crosstab(
        parcelas["data_real_pagamento"].isna(),
        parcelas["valor_pago_parcela"].isna(),
        rownames=["data_real_ausente"],
        colnames=["valor_pago_ausente"],
        dropna=False,
    )
)

In [ ]:
# Resume os registros por chave candidata, incluindo nulos como valores distintos.
resumo_chaves = (
    parcelas.groupby(chave_parcela, sort=False, observed=True)
    .agg(
        registros=("id_cliente", "size"),
        datas_previstas=("data_prevista_pagamento", "nunique"),
        valores_previstos=("valor_previsto_parcela", "nunique"),
        datas_reais=("data_real_pagamento", "nunique"),
        valores_pagos=("valor_pago_parcela", "nunique"),
    )
)

# Mostra quantos grupos possuem cada quantidade de registros.
display(
    resumo_chaves["registros"]
    .value_counts()
    .sort_index()
    .rename_axis("registros_por_chave")
    .to_frame("quantidade_chaves")
)

# Verifica mudanças no vencimento ou valor previsto dentro da mesma chave.
repetidas = resumo_chaves.loc[resumo_chaves["registros"].gt(1)]

print(f"Chaves repetidas: {len(repetidas):,}")
print(f"Com diferentes datas previstas: {repetidas['datas_previstas'].gt(1).sum():,}")
print(f"Com diferentes valores previstos: {repetidas['valores_previstos'].gt(1).sum():,}")

In [ ]:
# Resume os registros por chave candidata, incluindo nulos como valores distintos.
resumo_chaves = (
    parcelas.groupby(chave_parcela, sort=False, observed=True)
    .agg(
        registros=("id_cliente", "size"),
        datas_previstas=("data_prevista_pagamento", "nunique"),
        valores_previstos=("valor_previsto_parcela", "nunique"),
        datas_reais=("data_real_pagamento", "nunique"),
        valores_pagos=("valor_pago_parcela", "nunique"),
    )
)

# Mostra quantos grupos possuem cada quantidade de registros.
display(
    resumo_chaves["registros"]
    .value_counts()
    .sort_index()
    .rename_axis("registros_por_chave")
    .to_frame("quantidade_chaves")
)

# Verifica mudanças no vencimento ou valor previsto dentro da mesma chave.
repetidas = resumo_chaves.loc[resumo_chaves["registros"].gt(1)]

print(f"Chaves repetidas: {len(repetidas):,}")
print(f"Com diferentes datas previstas: {repetidas['datas_previstas'].gt(1).sum():,}")
print(f"Com diferentes valores previstos: {repetidas['valores_previstos'].gt(1).sum():,}")

In [ ]:
# Seleciona cinco chaves para inspeção, mantendo todas as colunas.
chaves_inspecao = repetidas.head(5).index.to_frame(index=False)

casos_repetidos = parcelas.merge(
    chaves_inspecao,
    on=chave_parcela,
    how="inner",
    validate="many_to_one",
)

with pd.option_context("display.max_columns", None):
    display(
        casos_repetidos.sort_values(
            chave_parcela + ["data_real_pagamento"]
        )
    )

In [ ]:
# Exibe registros com valor pago preenchido e data real ausente.
mascara_sem_data = (
    parcelas["data_real_pagamento"].isna()
    & parcelas["valor_pago_parcela"].notna()
)

with pd.option_context("display.max_columns", None):
    display(parcelas.loc[mascara_sem_data])

In [ ]:
parcelas[parcelas['id_contrato'] == 2331783]

In [ ]:
# Seleciona os contratos que possuem pagamento preenchido sem data.
contratos_sem_data = parcelas.loc[
    mascara_sem_data, "id_contrato"
].unique()

# Exibe todas as parcelas desses contratos, mantendo todas as colunas.
with pd.option_context("display.max_columns", None):
    display(
        parcelas.loc[
            parcelas["id_contrato"].isin(contratos_sem_data)
        ].sort_values(
            ["id_contrato", "versao_parcela", "numero_parcela",
             "data_real_pagamento"]
        )
    )

### Análise inicial — parcelas

- **Estrutura:** 1.664.384 registros e 8 colunas. Datas convertidas sem falhas, preservando ausências. Cada contrato está associado a um único cliente.
- **Versões:** variam de 0 a 52 e representam alterações da parcela. Não serão interpretadas automaticamente como renegociações.
- **Chave candidata:** `id_contrato + versao_parcela + numero_parcela`. Há 1.584.439 combinações distintas; 77.691 possuem múltiplas linhas, com vencimento e valor previsto iguais.
- **Pagamentos fracionados:** nos cinco grupos inspecionados, os pagamentos somam o valor previsto, com pagamento parcial antecipado e complementação após o vencimento. Chave repetida não significa duplicata descartável.
- **Duplicatas integrais:** 4.976 ocorrências adicionais de linhas idênticas. Podem inflar a soma dos pagamentos.
- **Ausências:** 3.652 registros sem data e valor pago; três com valor pago, mas sem data; nenhum com data e valor ausente.
- **Pagamentos elevados sem data:** os três casos superam o valor previsto e possuem parcelas posteriores pagas. Não há confirmação de antecipação ou quitação.

**Decisão:** preservar registros e valores, sem imputar datas. Validar versões, duplicatas e pagamentos elevados antes de consolidar parcelas e calcular atrasos. Ausência de pagamento não implica automaticamente inadimplência; futuras features devem respeitar o instante da previsão.



In [ ]:
# Resume valores monetários, versões e números de parcela.
colunas_numericas = [
    "versao_parcela",
    "numero_parcela",
    "valor_previsto_parcela",
    "valor_pago_parcela",
]

display(
    parcelas[colunas_numericas]
    .describe(percentiles=[0.01, 0.5, 0.95, 0.99])
    .T
)

# Conta ausências, zeros e negativos.
display(pd.DataFrame({
    "ausentes": parcelas[colunas_numericas].isna().sum(),
    "zeros": parcelas[colunas_numericas].eq(0).sum(),
    "negativos": parcelas[colunas_numericas].lt(0).sum(),
}))

# Verifica a cobertura temporal das datas.
colunas_datas = ["data_prevista_pagamento", "data_real_pagamento"]

display(pd.DataFrame({
    "data_minima": parcelas[colunas_datas].min(),
    "data_maxima": parcelas[colunas_datas].max(),
    "ausentes": parcelas[colunas_datas].isna().sum(),
}))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define o estilo e as variáveis numéricas de interesse.
sns.set_theme(style="whitegrid")

colunas = [
    "versao_parcela",
    "numero_parcela",
    "valor_previsto_parcela",
    "valor_pago_parcela",
]

fig, axes = plt.subplots(4, 2, figsize=(16, 16))

for linha, coluna in enumerate(colunas):
    # Exclui apenas nulos da visualização, sem modificar a base.
    serie = parcelas[coluna]
    valores = serie.dropna().to_numpy()

    # Usa intervalos de largura 1 para versões e números de parcela.
    if coluna in ["versao_parcela", "numero_parcela"]:
        bins = range(int(valores.min()), int(valores.max()) + 2)
    else:
        bins = 60

    # Mostra a distribuição completa.
    axes[linha, 0].hist(
        valores,
        bins=bins,
        color="#4C72B0",
        edgecolor="white",
    )
    axes[linha, 0].set_title(
        f"{coluna} | nulos: {serie.isna().sum():,} "
        f"| zeros: {serie.eq(0).sum():,}"
    )
    axes[linha, 0].set_xlabel(coluna)
    axes[linha, 0].set_ylabel("Frequência")

    # Mantém todos os extremos visíveis.
    axes[linha, 1].boxplot(
        valores,
        orientation="horizontal",
        flierprops={
            "marker": ".",
            "markersize": 2,
            "alpha": 0.3,
        },
    )
    axes[linha, 1].set_title(f"{coluna}: boxplot")
    axes[linha, 1].set_xlabel(coluna)
    axes[linha, 1].set_yticks([])

plt.tight_layout()
plt.show()
plt.close(fig)

## Distribuições numéricas e cobertura temporal

- **Versão:** concentração em 0 e 1; percentil 99 igual a 4 e máximo 52. Versões elevadas são raras, mas não foram classificadas como erros.
- **Número da parcela:** varia de 1 a 194, com mediana 8 e concentração nas primeiras parcelas. Representa a posição da parcela, não o prazo total do contrato.
- **Valores monetários:** forte assimetria à direita. Medianas de 8.841,47 para o valor previsto e 8.186,40 para o pago; máximos de aproximadamente 2,78 milhões e 6,56 milhões, respectivamente.
- **Zeros e ausências:** quatro valores previstos iguais a zero; 103 pagamentos iguais a zero e 3.652 ausentes. Não há valores negativos nas variáveis analisadas.
- **Cobertura temporal:** vencimentos entre 28/02/2017 e 22/02/2025; pagamentos entre 12/02/2017 e 22/02/2025. O limite observado não confirma a data de referência da extração.
- **Limitação:** as distribuições representam registros, incluindo pagamentos fracionados e duplicatas, não parcelas únicas. As medianas de valor previsto e pago não demonstram, isoladamente, pagamento insuficiente.

In [ ]:
# Calcula a diferença por registro; pagamentos ausentes permanecem nulos.
parcelas["diferenca_pago_previsto"] = (
    parcelas["valor_pago_parcela"]
    - parcelas["valor_previsto_parcela"]
)

# Resume a distribuição da diferença.
display(
    parcelas["diferenca_pago_previsto"]
    .describe(percentiles=[0.01, 0.5, 0.95, 0.99])
    .to_frame()
)

# Usa tolerância de um centavo para evitar diferenças residuais de ponto flutuante.
diferenca = parcelas["diferenca_pago_previsto"]

display(pd.Series({
    "abaixo_do_previsto": diferenca.lt(-0.01).sum(),
    "equivalente_ao_previsto": diferenca.abs().le(0.01).sum(),
    "acima_do_previsto": diferenca.gt(0.01).sum(),
    "ausentes": diferenca.isna().sum(),
}, name="quantidade").to_frame())

In [ ]:
# Inspeciona os 103 registros com valor pago igual a zero.
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(parcelas.loc[parcelas["valor_pago_parcela"].eq(0)])

In [ ]:
# Identifica as chaves que possuem valor pago ausente.
chaves_com_nulos = (
    parcelas.loc[parcelas["valor_pago_parcela"].isna(), chave_parcela]
    .drop_duplicates()
)

# Busca todos os registros dessas chaves, incluindo possíveis pagamentos.
historico_chaves_nulas = parcelas.merge(
    chaves_com_nulos,
    on=chave_parcela,
    how="inner",
    validate="many_to_one",
)

# Conta pagamentos preenchidos por chave.
pagamentos_por_chave = (
    historico_chaves_nulas.groupby(chave_parcela, sort=False)
    ["valor_pago_parcela"]
    .count()
)

display(pd.Series({
    "chaves_sem_valor_pago_preenchido": pagamentos_por_chave.eq(0).sum(),
    "chaves_com_algum_valor_pago_preenchido": pagamentos_por_chave.gt(0).sum(),
}, name="quantidade").to_frame())

In [ ]:
# Define a chave candidata explicitamente.
chave_parcela = ["id_contrato", "versao_parcela", "numero_parcela"]

# Seleciona as chaves com pelo menos um pagamento zerado.
chaves_com_zero = (
    parcelas.loc[parcelas["valor_pago_parcela"].eq(0), chave_parcela]
    .drop_duplicates()
)

# Recupera todas as linhas dessas mesmas chaves, sem consolidar valores.
historico_chaves_zero = parcelas.merge(
    chaves_com_zero,
    on=chave_parcela,
    how="inner",
    validate="many_to_one",
)

# Verifica se existe pagamento positivo na mesma chave.
tem_pagamento_positivo = (
    historico_chaves_zero["valor_pago_parcela"].gt(0)
    .groupby([
        historico_chaves_zero[coluna]
        for coluna in chave_parcela
    ], sort=False)
    .any()
)

print(pd.Series({
    "chaves_com_pagamento_positivo": tem_pagamento_positivo.sum(),
    "chaves_sem_pagamento_positivo": (~tem_pagamento_positivo).sum(),
}, name="quantidade").to_frame())

# Exibe todas as linhas das chaves selecionadas.
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(
        historico_chaves_zero.sort_values(
            chave_parcela + ["data_real_pagamento"]
        )
    )

In [ ]:
# Identifica as chaves com valor previsto zerado.
chaves_previsto_zero = (
    parcelas.loc[parcelas["valor_previsto_parcela"].eq(0), chave_parcela]
    .drop_duplicates()
)

# Recupera o histórico dessas chaves para contextualizar os zeros.
historico_previsto_zero = parcelas.merge(
    chaves_previsto_zero,
    on=chave_parcela,
    how="inner",
    validate="many_to_one",
)

with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(
        historico_previsto_zero.sort_values(
            chave_parcela + ["data_real_pagamento"]
        )
    )

In [ ]:
# Identifica contrato e número das parcelas com previsto zero.
chave_sem_versao = ["id_contrato", "numero_parcela"]

parcelas_previsto_zero = (
    historico_previsto_zero[chave_sem_versao]
    .drop_duplicates()
)

# Busca as mesmas parcelas em todas as versões.
historico_versoes_zero = parcelas.merge(
    parcelas_previsto_zero,
    on=chave_sem_versao,
    how="inner",
    validate="many_to_one",
)

# Exibe as versões para comparação, sem escolher uma versão vigente.
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(
        historico_versoes_zero.sort_values(
            chave_sem_versao + ["versao_parcela", "data_real_pagamento"]
        )
    )

In [ ]:
# Identifica duplicatas sem considerar a variável derivada.
duplicatas = parcelas.drop(columns="diferenca_pago_previsto").duplicated()
quantidade_removida = duplicatas.sum()
percentual_removido = duplicatas.mean() * 100

# Remove somente ocorrências adicionais na cópia de trabalho.
parcelas = parcelas.loc[~duplicatas].copy()

print(f"Removidas: {quantidade_removida:,} ({percentual_removido:.3f}%)")
print(f"Linhas restantes: {len(parcelas):,}")

## Zeros, nulos e remoção de duplicatas

- Pagamentos zerados possuem pagamentos positivos posteriores na mesma chave; não representam, isoladamente, quitação ou inadimplência.
- Quatro registros têm valor previsto zero e pagamento ausente, sem outras versões para o mesmo contrato e número de parcela. Permanecem sem classificação de atraso.
- Das 3.625 chaves com valor pago ausente, 247 também possuem algum valor pago preenchido, sem confirmação automática de quitação.
- Foram removidas 4.976 duplicatas integrais (0,299% das linhas), mantendo 1.659.408 registros na cópia de trabalho. A base bruta foi preservada.
- A identificação desconsiderou `diferenca_pago_previsto`. Chaves repetidas com pagamentos distintos foram mantidas.
- A remoção evita dupla contagem, mas assume que linhas integralmente idênticas são repetições, pois não há identificador de evento.
- A diferença pago menos previsto por linha é diagnóstica, não representa saldo da parcela.

In [ ]:
# --- 01_EDA: RESUMO DE PARCELAS POR CONTRATO (ALINHADO AO 02_TRANSFORM) ---

# 1. Parâmetros e data de corte factual
tolerancia_centavos = 0.01
chave_parcela = ["id_contrato", "numero_parcela"]
data_referencia = parcelas["data_real_pagamento"].max()

# 2. Versão mais recente de cada parcela
max_versoes = parcelas.groupby(chave_parcela)["versao_parcela"].transform("max")
parc_ativas = parcelas.loc[parcelas["versao_parcela"] == max_versoes].copy()

# Pagamento positivo sem data registrada (incerteza), marcado por linha antes de agregar
parc_ativas["_pgto_sem_data"] = (
    parc_ativas["valor_pago_parcela"].gt(0) & parc_ativas["data_real_pagamento"].isna()
)

# 3. Consolidação no grão (id_contrato, numero_parcela)
resumo_parcelas = (
    parc_ativas.groupby(chave_parcela)
    .agg(
        data_prevista_pagamento=("data_prevista_pagamento", "first"),
        valor_previsto_parcela=("valor_previsto_parcela", "first"),
        total_pago=("valor_pago_parcela", "sum"),
        ultima_data_pagamento=("data_real_pagamento", "max"),
        pagamento_sem_data=("_pgto_sem_data", "any"),
    )
    .reset_index()
)

# 4. Quitação e incerteza
resumo_parcelas["quitada"] = (
    resumo_parcelas["total_pago"]
    >= (resumo_parcelas["valor_previsto_parcela"] - tolerancia_centavos)
)
incerto = (
    resumo_parcelas["pagamento_sem_data"]
    | ~resumo_parcelas["valor_previsto_parcela"].gt(0)
    | resumo_parcelas["data_prevista_pagamento"].isna()
)

# 5. Data de liquidação e dias de atraso
resumo_parcelas["data_liquidacao"] = (
    resumo_parcelas["ultima_data_pagamento"].where(resumo_parcelas["quitada"])
    .fillna(resumo_parcelas["data_prevista_pagamento"].where(resumo_parcelas["quitada"]))
    .fillna(data_referencia)
)
dias_diferenca = (
    resumo_parcelas["data_liquidacao"] - resumo_parcelas["data_prevista_pagamento"]
).dt.days
resumo_parcelas["dias_atraso"] = dias_diferenca.clip(lower=0).astype("Int64").mask(incerto)

print(f"Data factual de corte (max data_real_pagamento): {data_referencia.strftime('%Y-%m-%d')}")
print(f"Total de prestações únicas avaliadas: {len(resumo_parcelas)}")
display(resumo_parcelas["dias_atraso"].describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]))

In [ ]:
resumo_parcelas

### Atraso de parcelas — cálculo exploratório

- A referência foi definida automaticamente como o maior vencimento da base. Essa escolha é arbitrária, apenas para termos um parâmetro fixo.
- Para parcelas quitadas, o atraso considera a primeira data em que os pagamentos acumulados atingem o valor previsto, com tolerância provisória de R$ 0,01. Para as demais, considera o intervalo entre vencimento e referência, com mínimo zero.
- O cálculo mantém versões separadas e pressupõe que parcelas não quitadas continuem exigíveis. Targets definitivas dependem da validação dessa hipótese e da data de referência.
- Foram calculados atrasos para 1.584.432 parcelas/versões. A mediana é zero, o p95 é 8 dias, o p99 é 45 dias e o máximo é 2.893 dias. Os extremos foram mantidos.
- Sete parcelas/versões permaneceram com atraso nulo: três possuem pagamento positivo sem data e quatro possuem valor previsto zero.
- Esses casos foram preservados, sem imputação de atraso zero e sem classificação automática de inadimplência. Nas futuras targets, não devem ser tratados automaticamente como negativos; seu impacto será avaliado na consolidação por contrato.

In [ ]:
# --- DIAGNÓSTICO: VERSÕES DE PARCELAS - Renegociação de Contratos ---

chave_parcela = ["id_contrato", "numero_parcela"]
tolerancia_centavos = 0.01

df_v = parcelas[
    chave_parcela
    + ["versao_parcela", "data_prevista_pagamento", "data_real_pagamento",
       "valor_previsto_parcela", "valor_pago_parcela"]
].copy()

# 1. Marcações no nível de linha
grupo = df_v.groupby(chave_parcela)
df_v["_n_versoes"] = grupo["versao_parcela"].transform("nunique")
df_v["_eh_maxima"] = df_v["versao_parcela"].eq(grupo["versao_parcela"].transform("max"))
df_v["_pago_positivo"] = df_v["valor_pago_parcela"].gt(0)

# 2. Prevalência de múltiplas versões
n_parcelas = df_v[chave_parcela].drop_duplicates().shape[0]
parcelas_multi = df_v.loc[df_v["_n_versoes"] > 1, chave_parcela].drop_duplicates()
contratos_multi = parcelas_multi["id_contrato"].nunique()

print("--- Prevalência de múltiplas versões ---")
print(f"Parcelas únicas: {n_parcelas}")
print(f"Parcelas com mais de uma versão: {len(parcelas_multi)} ({len(parcelas_multi) / n_parcelas:.4%})")
print(f"Contratos afetados: {contratos_multi} ({contratos_multi / df_v['id_contrato'].nunique():.4%})")
print("Distribuição do nº de versões por parcela:")
print(df_v.drop_duplicates(chave_parcela)["_n_versoes"].value_counts().sort_index())

# 3. Onde está o dinheiro pago: versão máxima vs demais
pago_total = df_v["valor_pago_parcela"].sum()
pago_fora_max = df_v.loc[~df_v["_eh_maxima"], "valor_pago_parcela"].sum()

print("\n--- Valor pago fora da versão máxima ---")
print(f"Linhas fora da versão máxima: {(~df_v['_eh_maxima']).sum()}")
print(f"Linhas fora da máxima com pagamento > 0: {(~df_v['_eh_maxima'] & df_v['_pago_positivo']).sum()}")
print(f"% do valor pago total fora da versão máxima: {pago_fora_max / pago_total:.4%}")

# 4. Pagamentos idênticos repetidos entre versões (risco de dupla contagem se somar todas)
colunas_pgto = chave_parcela + ["data_real_pagamento", "valor_pago_parcela"]
pgto_repetido = (
    df_v["_pago_positivo"]
    & df_v["_n_versoes"].gt(1)
    & df_v.duplicated(subset=colunas_pgto, keep=False)
)
print(f"\nLinhas de pagamento idêntico repetido em versões distintas: {pgto_repetido.sum()}")

# 5. As versões divergem em vencimento ou valor previsto?
prev_distintos = grupo["data_prevista_pagamento"].transform("nunique").gt(1)
valor_distintos = grupo["valor_previsto_parcela"].transform("nunique").gt(1)
print("\n--- Divergência entre versões da mesma parcela ---")
print(f"Parcelas com data prevista diferente entre versões: {df_v.loc[prev_distintos, chave_parcela].drop_duplicates().shape[0]}")
print(f"Parcelas com valor previsto diferente entre versões: {df_v.loc[valor_distintos, chave_parcela].drop_duplicates().shape[0]}")

# 6. Impacto direto na quitação: somar só a versão máxima vs somar todas
total_todas = grupo["valor_pago_parcela"].sum()
total_max = df_v.loc[df_v["_eh_maxima"]].groupby(chave_parcela)["valor_pago_parcela"].sum()
previsto_max = df_v.loc[df_v["_eh_maxima"]].groupby(chave_parcela)["valor_previsto_parcela"].first()

comparativo = pd.DataFrame({
    "total_todas": total_todas,
    "total_max": total_max,
    "previsto_max": previsto_max,
})
limite_quitacao = comparativo["previsto_max"] - tolerancia_centavos
comparativo["quitada_todas"] = comparativo["total_todas"] >= limite_quitacao
comparativo["quitada_max"] = comparativo["total_max"] >= limite_quitacao

so_todas = comparativo["quitada_todas"] & ~comparativo["quitada_max"]
print("\n--- Impacto na quitação ---")
print(f"Parcelas quitadas somando todas as versões: {comparativo['quitada_todas'].sum()}")
print(f"Parcelas quitadas somando só a versão máxima: {comparativo['quitada_max'].sum()}")
print(f"Quitadas apenas se somar todas as versões: {so_todas.sum()}")
print(f"Contratos com ao menos uma parcela nessa condição: {so_todas[so_todas].reset_index()['id_contrato'].nunique()}")


In [ ]:
# --- GRÁFICO: VARIAÇÃO DO VALOR PREVISTO ENTRE PRIMEIRA E ÚLTIMA VERSÃO ---

chave_parcela = ["id_contrato", "numero_parcela"]
percentil_inf, percentil_sup = 0.01, 0.99  # recorte apenas para visualização
n_bins = 50

# 1. Parcelas com mais de uma versão, ordenadas por versão
multi = df_v.loc[df_v["_n_versoes"] > 1].sort_values(chave_parcela + ["versao_parcela"])

# 2. Valor previsto na primeira e na última versão de cada parcela
g_valor = multi.groupby(chave_parcela)["valor_previsto_parcela"]
variacao = pd.DataFrame({"primeira": g_valor.first(), "ultima": g_valor.last()})

# 3. Variação percentual (denominador não positivo vira nulo)
variacao["var_pct"] = (
    variacao["ultima"] / variacao["primeira"].where(variacao["primeira"] > 0) - 1
) * 100

serie = variacao["var_pct"].dropna()
limite_inf, limite_sup = serie.quantile([percentil_inf, percentil_sup])

print(f"Parcelas analisadas: {len(serie)}")
print(f"Sem variação de valor: {(serie == 0).mean():.2%} | Redução: {(serie < 0).mean():.2%} | Aumento: {(serie > 0).mean():.2%}")
print(serie.describe())

# 4. Histograma
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(serie.clip(limite_inf, limite_sup), bins=n_bins, color="steelblue", edgecolor="white")
ax.axvline(0, color="red", linestyle="--", linewidth=1)
ax.set_title("Variação do valor previsto: primeira vs. última versão da parcela")
ax.set_xlabel(f"Variação % (recortado entre p{percentil_inf:.0%} e p{percentil_sup:.0%})")
ax.set_ylabel("Nº de parcelas")
plt.tight_layout()
plt.show()

In [ ]:
# --- 01_EDA: MATRIZ TEMPORAL OVER X MOB (ALINHADA AO 02_TRANSFORM) ---

# 1. Parâmetros
limites_dpd = (30, 60, 90)
janelas_mob = (6, 9, 12)
dias_mes_mob = 30
chave_parcela = ["id_contrato", "numero_parcela"]

# 2. Versão máxima já aplicada na construção do resumo_parcelas; valida a unicidade da chave
assert not resumo_parcelas.duplicated(chave_parcela).any(), (
    "Chave id_contrato + numero_parcela não é única em resumo_parcelas"
)

# 3. Origem fixa em data_decisao, sem fallback
origem_contrato = (
    emprestimos[["id_contrato", "data_decisao"]]
    .drop_duplicates("id_contrato")
    .rename(columns={"data_decisao": "data_origem"})
)
base_mob = resumo_parcelas.merge(origem_contrato, on="id_contrato", how="left")
assert base_mob["data_origem"].notna().all(), "Há contratos de parcelas sem data_decisao em emprestimos"

# 4. Contratos únicos e estruturas de saída
contratos_unicos = (
    base_mob[["id_contrato", "data_origem"]]
    .drop_duplicates("id_contrato")
    .set_index("id_contrato")
)
targets_mob = pd.DataFrame(index=contratos_unicos.index)
resultados_tabela = []

print(f"Data de corte (data_referencia): {data_referencia}")

# 5. Varredura de limiares de DPD e janelas MOB
for limite in limites_dpd:
    data_limiar_parcela = base_mob["data_prevista_pagamento"] + pd.Timedelta(days=limite)
    atingiu_limite = base_mob["dias_atraso"].ge(limite)

    for mob in janelas_mob:
        nome_metrica = f"OVER{limite}MOB{mob}"
        dias_janela = dias_mes_mob * mob
        fim_janela_parcela = base_mob["data_origem"] + pd.Timedelta(days=dias_janela)
        limiar_na_janela = data_limiar_parcela <= fim_janela_parcela

        evento_no_mob = atingiu_limite & limiar_na_janela
        incerta_no_mob = base_mob["dias_atraso"].isna() & limiar_na_janela

        teve_evento = evento_no_mob.groupby(base_mob["id_contrato"]).any().reindex(contratos_unicos.index)
        teve_incerteza = incerta_no_mob.groupby(base_mob["id_contrato"]).any().reindex(contratos_unicos.index)

        janela_maturada = (contratos_unicos["data_origem"] + pd.Timedelta(days=dias_janela)) <= data_referencia

        # Variante "fora": 0 = maduro sem incerteza; 1 = maduro com evento; <NA> = demais
        target_serie = pd.Series(pd.NA, index=contratos_unicos.index, dtype="Int8")
        target_serie = target_serie.mask(janela_maturada & ~teve_incerteza, 0)
        target_serie = target_serie.mask(janela_maturada & teve_evento, 1)

        targets_mob[nome_metrica] = target_serie

        avaliaveis = int(target_serie.notna().sum())
        positivos = int(target_serie.eq(1).sum())
        resultados_tabela.append({
            "metrica": nome_metrica,
            "avaliaveis": avaliaveis,
            "positivos": positivos,
            "indeterminados": int(target_serie.isna().sum()),
            "taxa_positivos": positivos / avaliaveis * 100 if avaliaveis > 0 else np.nan,
        })

tabela_over_mob = pd.DataFrame(resultados_tabela).set_index("metrica")
display(tabela_over_mob)

In [ ]:
# --- ROLLUP DE PARCELAS E TARGET OVER60MOB12: VARIANTES DENTRO E FORA do período 10+ meses---

# 1. Parâmetros
tolerancia_centavos = 0.01
limite_dias_atraso = 60
dias_mes_mob = 30
multiplicador_mob = 12
dias_janela_mob = dias_mes_mob * multiplicador_mob  # 360 dias
filtrar_versao_maxima = True  # ajustar conforme o resultado do diagnóstico de versões

chave_parcela = ["id_contrato", "numero_parcela"]
n_contratos_esperados_parc = parcelas["id_contrato"].nunique()
data_corte_sistema = parcelas["data_real_pagamento"].max()

# 2. Seleção de versão (parametrizada)
if filtrar_versao_maxima:
    versao_max = parcelas.groupby(chave_parcela)["versao_parcela"].transform("max")
    parc_base = parcelas.loc[parcelas["versao_parcela"] == versao_max].copy()
else:
    parc_base = parcelas.copy()

# Pagamento positivo sem data (incerteza), marcado por linha antes de agregar
parc_base["_pgto_sem_data"] = (
    parc_base["valor_pago_parcela"].gt(0) & parc_base["data_real_pagamento"].isna()
)

# 3. Consolidação no grão (id_contrato, numero_parcela)
resumo_parcelas = (
    parc_base.groupby(chave_parcela)
    .agg(
        data_prevista_pagamento=("data_prevista_pagamento", "first"),
        valor_previsto_parcela=("valor_previsto_parcela", "first"),
        total_pago=("valor_pago_parcela", "sum"),
        ultima_data_pagamento=("data_real_pagamento", "max"),
        pagamento_sem_data=("_pgto_sem_data", "any"),
    )
    .reset_index()
)

# 4. Quitação e incerteza
resumo_parcelas["quitada"] = (
    resumo_parcelas["total_pago"]
    >= (resumo_parcelas["valor_previsto_parcela"] - tolerancia_centavos)
)
incerto = (
    resumo_parcelas["pagamento_sem_data"]
    | ~resumo_parcelas["valor_previsto_parcela"].gt(0)
    | resumo_parcelas["data_prevista_pagamento"].isna()
)

# 5. Data de liquidação e dias de atraso
data_liquidacao = (
    resumo_parcelas["ultima_data_pagamento"].where(resumo_parcelas["quitada"])
    .fillna(resumo_parcelas["data_prevista_pagamento"].where(resumo_parcelas["quitada"]))
    .fillna(data_corte_sistema)
)
dias_diferenca = (data_liquidacao - resumo_parcelas["data_prevista_pagamento"]).dt.days
resumo_parcelas["dias_atraso"] = dias_diferenca.clip(lower=0).astype("Int64").mask(incerto)

# 6. Origem do contrato: data_decisao (intermediária, não persistida)
origem_por_contrato = emprestimos.set_index("id_contrato")["data_decisao"]
resumo_parcelas["_data_origem"] = resumo_parcelas["id_contrato"].map(origem_por_contrato)
assert resumo_parcelas["_data_origem"].notna().all(), (
    "Há contratos de parcelas sem data_decisao em emprestimos"
)

# 7. Eventos dentro da janela MOB 12
limiar_dentro_janela = (
    resumo_parcelas["data_prevista_pagamento"] + pd.Timedelta(days=limite_dias_atraso)
) <= (resumo_parcelas["_data_origem"] + pd.Timedelta(days=dias_janela_mob))

evento_no_mob = resumo_parcelas["dias_atraso"].ge(limite_dias_atraso) & limiar_dentro_janela
incerta_no_mob = resumo_parcelas["dias_atraso"].isna() & limiar_dentro_janela

# 8. Agregação por contrato
id_contrato_parcela = resumo_parcelas["id_contrato"]
teve_evento = evento_no_mob.groupby(id_contrato_parcela).any()
teve_incerteza = incerta_no_mob.groupby(id_contrato_parcela).any()

origem_contrato = resumo_parcelas.groupby("id_contrato")["_data_origem"].first()
janela_maturada = (origem_contrato + pd.Timedelta(days=dias_janela_mob)) <= data_corte_sistema

# 9. Variante DENTRO: positivo assim que o evento ocorre, mesmo em janela imatura
target_dentro = pd.Series(pd.NA, index=origem_contrato.index, dtype="Int8")
target_dentro = target_dentro.mask(janela_maturada & ~teve_incerteza, 0)
target_dentro = target_dentro.mask(teve_evento, 1)

# 10. Variante FORA: positivos de janela imatura ficam <NA>
target_fora = pd.Series(pd.NA, index=origem_contrato.index, dtype="Int8")
target_fora = target_fora.mask(janela_maturada & ~teve_incerteza, 0)
target_fora = target_fora.mask(janela_maturada & teve_evento, 1)

# 11. Tabela de contratos: apenas chave e targets candidatos
parcelas_por_contrato = pd.DataFrame({
    "OVER60MOB12_dentro": target_dentro,
    "OVER60MOB12_fora": target_fora,
}).reset_index()

# 12. Asserções
assert len(parcelas_por_contrato) == n_contratos_esperados_parc, (
    f"Esperado {n_contratos_esperados_parc} contratos, obtido {len(parcelas_por_contrato)}"
)
assert parcelas_por_contrato["id_contrato"].is_unique, "Falha de unicidade em id_contrato"
assert not any(c.startswith("_") for c in parcelas_por_contrato.columns), "Coluna intermediária vazou"

# 13. Comparativo das variantes
linhas = []
for nome, serie in (("dentro", target_dentro), ("fora", target_fora)):
    n_bons = int(serie.eq(0).sum())
    n_maus = int(serie.eq(1).sum())
    linhas.append({
        "variante": nome,
        "bons": n_bons,
        "maus": n_maus,
        "indeterminados": int(serie.isna().sum()),
        "avaliaveis": n_bons + n_maus,
        "taxa_maus_%": n_maus / (n_bons + n_maus) * 100,
    })

print(f"Contratos consolidados: {len(parcelas_por_contrato)}")
print(f"Contratos com janela madura: {int(janela_maturada.sum())}")
print(f"Positivos em janela imatura (removidos na variante 'fora'): {int((teve_evento & ~janela_maturada).sum())}")
display(pd.DataFrame(linhas).set_index("variante"))

In [ ]:
# --- ROLL RATE REAL (+30 dias para 30->60 e +60 dias para 60->90), SOBRE A MESMA base_mob ---

# 1. Data em que cada parcela cruzou cada limiar (NaT se não cruzou), derivada de dias_atraso
venc = base_mob["data_prevista_pagamento"]
data_30 = (venc + pd.Timedelta(days=30)).where(base_mob["dias_atraso"].ge(30))
data_60 = (venc + pd.Timedelta(days=60)).where(base_mob["dias_atraso"].ge(60))
data_90 = (venc + pd.Timedelta(days=90)).where(base_mob["dias_atraso"].ge(90))

resumo_roll_real = []

for mob in janelas_mob:
    fim_mob_30 = base_mob["data_origem"] + pd.Timedelta(days=dias_mes_mob * mob)
    fim_mob_60 = base_mob["data_origem"] + pd.Timedelta(days=dias_mes_mob * (mob + 1))
    fim_mob_90 = base_mob["data_origem"] + pd.Timedelta(days=dias_mes_mob * (mob + 2))

    # Comparações com NaT resultam em False
    ev_30 = data_30.le(fim_mob_30)
    ev_60 = ev_30 & data_60.le(fim_mob_60)
    ev_90 = ev_60 & data_90.le(fim_mob_90)

    # Agregação por contrato
    c_ev30 = ev_30.groupby(base_mob["id_contrato"]).any().reindex(contratos_unicos.index)
    c_ev60 = ev_60.groupby(base_mob["id_contrato"]).any().reindex(contratos_unicos.index)
    c_ev90 = ev_90.groupby(base_mob["id_contrato"]).any().reindex(contratos_unicos.index)

    # Maturação até fim_mob_90, para não truncar a rolagem
    maturado_ate_90 = (
        contratos_unicos["data_origem"] + pd.Timedelta(days=dias_mes_mob * (mob + 2))
    ) <= data_referencia

    n_30 = int(c_ev30.loc[maturado_ate_90].sum())
    n_60 = int(c_ev60.loc[maturado_ate_90].sum())
    n_90 = int(c_ev90.loc[maturado_ate_90].sum())

    resumo_roll_real.append({
        "coorte_entrada": f"Entrou 30+ até MOB{mob}",
        "contratos_maturados": int(maturado_ate_90.sum()),
        "entraram_30mais": n_30,
        "evoluiram_60mais": n_60,
        "evoluiram_90mais": n_90,
        "roll_30_para_60_pct": n_60 / n_30 * 100 if n_30 else np.nan,
        "roll_60_para_90_pct": n_90 / n_60 * 100 if n_60 else np.nan,
        "roll_30_para_90_pct": n_90 / n_30 * 100 if n_30 else np.nan,
    })

display(pd.DataFrame(resumo_roll_real).set_index("coorte_entrada"))

In [ ]:
# --- GRÁFICO: TAXA DE INADIMPLÊNCIA POR OVER (EIXO Y) E MOB (EIXO X) ---
import matplotlib.pyplot as plt
import numpy as np

# 1. Parâmetros de apresentação
ordem_over = [30, 60, 90]
ordem_mob = [6, 9, 12]
mapa_cores = "YlOrRd"

# 2. Extrai limiar (OVER) e janela (MOB) do nome da métrica, ex.: OVER60MOB12
padrao = tabela_over_mob.index.to_series().str.extract(r"OVER(?P<over>\d+)MOB(?P<mob>\d+)")
base_plot = tabela_over_mob[["taxa_positivos"]].copy()
base_plot["over"] = padrao["over"].astype(int)
base_plot["mob"] = padrao["mob"].astype(int)

# 3. Matriz OVER x MOB, com linhas e colunas em ordem explícita
matriz = (
    base_plot.pivot(index="over", columns="mob", values="taxa_positivos")
    .reindex(index=ordem_over, columns=ordem_mob)
)

assert not matriz.isna().any().any(), "Há combinações OVER x MOB ausentes na tabela_over_mob"

# 4. Heatmap com a taxa anotada em cada célula
fig, ax = plt.subplots(figsize=(7, 4.5), dpi=120)
im = ax.imshow(matriz.to_numpy(), cmap=mapa_cores, aspect="auto")

ax.set_xticks(np.arange(len(ordem_mob)))
ax.set_xticklabels([f"MOB {m}" for m in ordem_mob])
ax.set_yticks(np.arange(len(ordem_over)))
ax.set_yticklabels([f"OVER {o}+" for o in ordem_over])
ax.set_xlabel("Janela de observação (MOB)")
ax.set_ylabel("Limiar de atraso (OVER)")
ax.set_title("Taxa de inadimplência por OVER x MOB", fontsize=12, fontweight="bold", pad=12)

# Cor do texto contrasta com o fundo da célula
limite_contraste = np.nanmax(matriz.to_numpy()) * 0.6
for i in range(matriz.shape[0]):
    for j in range(matriz.shape[1]):
        valor = matriz.iloc[i, j]
        cor_texto = "white" if valor > limite_contraste else "black"
        ax.text(j, i, f"{valor:.2f}%", ha="center", va="center",
                fontsize=11, fontweight="bold", color=cor_texto)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Taxa de inadimplência (%)")

plt.tight_layout()
plt.show()

In [ ]:
# --- GRÁFICO: FUNIL DA ROLL RATE (30+ -> 60+ -> 90+) POR COORTE MOB ---
import matplotlib.pyplot as plt
import numpy as np

# 1. Dados da Roll Rate recalculada sob a regra atual
df_roll = pd.DataFrame(resumo_roll_real).set_index("coorte_entrada")
coortes = ["Até MOB 6", "Até MOB 9", "Até MOB 12"]

assert len(df_roll) == len(coortes), "Número de coortes diferente do esperado"

# 2. Etapas do funil: contagens e taxas de rolagem entre etapas
contagens = {
    "30+": df_roll["entraram_30mais"].to_numpy(),
    "60+": df_roll["evoluiram_60mais"].to_numpy(),
    "90+": df_roll["evoluiram_90mais"].to_numpy(),
}
taxas = {
    "60+": df_roll["roll_30_para_60_pct"].to_numpy(),  # 30+ -> 60+
    "90+": df_roll["roll_60_para_90_pct"].to_numpy(),  # 60+ -> 90+
}
cores = {"30+": "#2b5c8f", "60+": "#e08214", "90+": "#b2182b"}

# 3. Barras agrupadas por coorte
x = np.arange(len(coortes))
largura = 0.26
deslocamentos = {"30+": -largura, "60+": 0.0, "90+": largura}

fig, ax = plt.subplots(figsize=(10, 5.5), dpi=120)

for etapa, valores in contagens.items():
    rects = ax.bar(x + deslocamentos[etapa], valores, width=largura,
                   label=f"Atingiu {etapa} dias", color=cores[etapa], alpha=0.9)

    for i, rect in enumerate(rects):
        altura = rect.get_height()
        # Contagem absoluta e, nas etapas 60+ e 90+, a taxa de rolagem vinda da etapa anterior
        texto = f"{int(altura):,}".replace(",", ".")
        if etapa in taxas:
            texto += f"\n({taxas[etapa][i]:.1f}% da etapa anterior)"
        ax.annotate(texto,
                    xy=(rect.get_x() + rect.get_width() / 2, altura),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8.5, fontweight="bold")

# 4. Formatação
ax.set_xticks(x)
ax.set_xticklabels(coortes)
ax.set_ylabel("Contratos")
ax.set_title("Roll Rate: funil de rolagem 30+ → 60+ → 90+ por coorte",
             fontsize=12, fontweight="bold", pad=12)
ax.set_ylim(0, contagens["30+"].max() * 1.2)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, loc="upper left")

plt.tight_layout()
plt.show()

### Decisão: `valor_credito` (erro de escala)

Problema: 63 linhas com razão crédito/bem entre 80 e 200 (máx. 152,8); nenhuma entre 5 e 80. Bem e parcela dessas linhas são normais; só o crédito está inflado (ex.: 135.000.000 com bem de 1.350.000).
Validação no histórico: 62 dos 63 clientes têm histórico (266 contratos). Nos 161.422 contratos históricos com bem, a razão crédito/bem tem mediana 1,00 e máximo 6,67; nenhum acima de 80. O padrão não se repete no histórico, o que contraria a hipótese de valores reais.
Regra: dividir por 100 e criar a flag `credito_corrigido`; valor original em `valor_credito_original`.
Resultado: 63 linhas corrigidas, 50.400 linhas preservadas.
Não corrigido: 360 linhas com razão entre 1,5 e 5 e 12 abaixo de 0,5, plausíveis; vão para o cliente.
Limitações: o critério depende de `valor_bem`, então erros nas 2.460 linhas sem bem não são detectados por ele. Após a correção, a razão sobre a mediana histórica do cliente não se concentra em 1 (a mediana estimada fica em torno de 4), sem grupo de controle para comparar.
Para o cliente: o crédito de 63 contratos foi registrado com escala 100x? Razões acima de 1,5 e abaixo de 0,5 são legítimas?

### Decisão: `data_solicitacao`

Problema: 50 datas em `DD/MM/AAAA` (1º campo > 12, sem ambiguidade) e 51 datas com o valor `2025-13-01`, mês inexistente e idêntico em todas as linhas (provável placeholder).
Regra: conversão com `converter_tipos` (formatos ISO e `DD/MM/AAAA`); datas inválidas viram nulo; flag `data_invalida` criada a partir do bruto; sem imputação.
Resultado: 51 nulos (0,10%), período 04 a 24/02/2025, 50.400 linhas preservadas.
Para o cliente: o que significa `2025-13-01`?

### Decisão: `dia_semana_solicitacao`

Problema: a coluna original era redundante com `data_solicitacao`, e a comparação entre as duas não foi feita.
Regra: remover a coluna original; criar `dia_semana` derivado da data.
Resultado: 51 nulos (os das datas inválidas), 50.400 linhas preservadas.
Limitação: nas 51 linhas sem data, o dia da semana original foi descartado; consistência da coluna original não verificada.
Para o cliente: a coluna `dia_semana_solicitacao` era gerada a partir da data ou registrada de forma independente?

## Análise final - EDA das bases individuais

| Passo | Base / Coluna | Problema | Decisão | Resultado |
|---|---|---|---|---|
| 1 | `cadastral`: categóricas (`sexo`, `tipo_renda`, `possui_carro`, `possui_imovel`, etc.) | Variantes de grafia e placeholder `"Não informado"` | Padronizar strings; `"não informado"` vira nulo em todas (incluindo `tipo_renda`); excluir `sexo` do modelo | 50.600 linhas; 51, 60, 49 e 54 nulos nas 4 colunas |
| 2 | `cadastral`: `data_nascimento` | 2 formatos; 4 datas sentinela (`31/02/1990`, `1901-05-20`, `2030-01-01`, `2012-07-15` — 151 linhas) | Converter; sentinelas viram nulo; flag `data_nascimento_invalida`; idade calculada pós-*merge* na data do contrato (`2025-02-24` usada só como ref. exploratória) | 151 nulos (0,30%), 50.600 linhas |
| 3 | `cadastral`: `renda_anual`, `qtd_membros_familia`, `qtd_filhos` | `renda_anual` em texto BR (zeros, 67 negativos, >1M); zeros em `qtd_membros_familia`; correlação com `qtd_filhos` | Converter tipos sem alterar valores numéricos; manter `qtd_filhos` e `qtd_membros_familia` para avaliar sinal preditivo na modelagem | 50.600 linhas preservadas; `Int64` e `float64` |
| 4 | `submissao`: `tipo_contrato` | 6 grafias para 3 categorias (caixa e acento, 302 linhas) | Padronizar via `padronizar_categoricas` | 3 categorias, 0 nulos, 50.400 linhas |
| 5 | `submissao`: `valor_parcela`, `valor_bem`, `valor_credito` | 67 parcelas zero; 2.460 bens nulos; 63 razões crédito/bem entre 80 e 200 | Base simula dados de produção: **não alterar nem imputar valores numéricos**; criar apenas flags de alerta para uso no modelo e nas travas da política de crédito | 50.400 linhas com valores numéricos originais preservados |
| 6 | `submissao`: `data_solicitacao` e `dia_semana_solicitacao` | 50 datas em `DD/MM/AAAA`; 51 com string fixa `2025-13-01` | Converter formatos válidos; transformar `2025-13-01` em nulo com flag `data_invalida`; preservar `dia_semana_solicitacao` original | 51 datas nulas (0,10%), 50.400 linhas |
| 7 | `emprestimos`: categóricas, datas, `razao_credito_bem` e `motivo_recusa` | 16 categóricas despadronizadas; 6 datas em texto; 204 inversões de datas; 30 razões > 2 (29 recusados) | Padronizar strings e datas; manter valores numéricos e datas sem alterar; vetar `motivo_recusa` atual como feature | 211.944 linhas preservadas, 47.952 clientes |
| 8 | `parcelas`: deduplicação, versões e cálculo de atraso | 1.664.384 linhas brutas com 4.976 duplicatas integrais; pagamentos fracionados; 10.023 parcelas com múltiplas versões | Remover duplicatas integrais na cópia; manter somente a versão máxima de cada parcela; acumular pagamentos por `id_contrato + numero_parcela` (tolerância R$ 0,01); atraso calculado na quitação ou contra `max(data_real_pagamento)` | 1.659.408 linhas após deduplicação (0,299% removidas) → 1.574.411 parcelas únicas → 122.867 contratos; 7 atrasos nulos preservados |
| 9 | `parcelas` + `emprestimos`: Targets `OVER x MOB`, Roll Rate e Recusados | Diluição no nível parcela; 36.359 recusados sem parcelas; censura de janela | Consolidar por `id_contrato`; origem em `data_decisao`; janela MOB 12 = 360 dias; adotar `OVER60MOB12` (variante "fora") como target; não maturados e incertos como `<NA>` | `OVER60MOB12`: 104.157 avaliáveis, 7.446 positivos (7,15%), 18.710 indeterminados|

---

### Decisão: `cadastral` (padronização, idade pós-merge e manutenção de variáveis familiares)

- **Problema:** placeholders `"Não informado"` nas categóricas (incluindo 60 em `tipo_renda`), 151 datas de nascimento sentinela, `renda_anual` em formato BR com negativos/zeros e sobreposição entre `qtd_filhos` e `qtd_membros_familia`.
- **Regra:** normalizar strings e converter `"não informado"` em `<NA>` em todas as categóricas; converter `data_nascimento` tornando sentinelas nulas com flag `data_nascimento_invalida`. A data `pd.Timestamp("2025-02-24")` foi usada exclusivamente como referência exploratória; no `02_transform`, a idade será calculada na data de solicitação/decisão do contrato pós-*merge*. Preservar todos os valores numéricos originais e manter tanto `qtd_filhos` quanto `qtd_membros_familia` para avaliar qual apresenta melhor sinal preditivo nos modelos de árvore.
- **Resultado:** 50.600 linhas preservadas (600 linhas excedentes por `id_cliente` serão tratadas antes do *merge*).

### Decisão: `submissao` como base de produção (`valor_parcela`, `valor_bem`, `valor_credito`)

- **Problema:** 67 registros com `valor_parcela == 0`, 2.460 nulos em `valor_bem` (4,88%) e 63 linhas com razão crédito/bem entre 80 e 200 (ausentes no histórico de `emprestimos`, onde o máximo é 6,67).
- **Regra:** como `submissao` representa o fluxo de novas propostas para inferência e decisão em produção, **nenhum valor numérico será imputado ou editado**. Os dados originais são preservados integralmente para que o pipeline de inferência e as travas da Política de Crédito lidem diretamente com anomalias operacionais (ex.: sinalizar propostas com `valor_parcela == 0` ou `razao_credito_bem` extrema na regra de concessão). Criam-se apenas flags indicadoras (`valor_bem_ausente`, `alerta_escala_credito`).
- **Resultado:** 50.400 linhas com 100% dos valores numéricos originais preservados.

### Decisão: `data_solicitacao` e `dia_semana_solicitacao` (`submissao`)

- **Problema:** 50 datas em `DD/MM/AAAA` e 51 ocorrências idênticas da string `2025-13-01` (fora da janela de 04 a 24/02/2025 do restante da base).
- **Regra:** converter formatos válidos (`ISO` e `DD/MM/AAAA`) e transformar o placeholder `2025-13-01` em `<NA>` com flag `data_invalida`, sem imputar; manter `dia_semana_solicitacao` padronizada para não perder a informação de dia da semana nos 51 registros com data nula.
- **Resultado:** 51 datas nulas (0,10%), 50.400 linhas preservadas.

### Decisão: `emprestimos` (histórico, datas, `razao_credito_bem` e recusas)

- **Problema:** 16 colunas categóricas despadronizadas, 6 colunas de data em string, 204 inversões cronológicas (6 com liberação < decisão e 198 com 1º vencimento < decisão) e 30 contratos com `razao_credito_bem > 2` (29 deles recusados).
- **Regra:** padronizar categóricas e converter datas; não alterar nem excluir inversões de datas (aguardar validação sobre renegociações) nem extremos de `razao_credito_bem`; proibir `motivo_recusa` da proposta atual como feature (vazamento de decisão).
- **Resultado:** 211.944 contratos únicos preservados (47.952 clientes), 0 falhas adicionais de conversão.

### Decisão: `parcelas` (deduplicação, versões de parcela e cálculo de atraso)

- **Problema:** 1.664.384 linhas brutas contendo 4.976 duplicatas exatas em todas as 8 colunas (0,299%), além de múltiplos registros legítimos na mesma chave por pagamentos fracionados, 3 pagamentos positivos sem data e 4 previstos zerados. Há ainda 10.023 parcelas (0,64% das 1.574.411 únicas por `id_contrato + numero_parcela`) com mais de uma versão (10.018 com 2 versões e 5 com 3), afetando 5.847 contratos (4,76%). Existem 10.074 linhas fora da versão máxima, 10.043 delas com pagamento positivo, somando 4,71% do valor pago total, e 19.684 linhas de pagamento idêntico repetido entre versões. A data prevista difere entre versões em apenas 31 parcelas, enquanto o valor previsto difere em 10.018.
- **Regra:** remover as 4.976 duplicatas integrais na cópia de trabalho; manter somente a versão máxima de cada parcela (`filtrar_versao_maxima = True`), pois somar todas as versões duplicaria pagamentos repetidos (hipótese coerente com as 19.684 linhas idênticas, não confirmada) e o impacto na quitação é desprezível; agregar pagamentos acumulados por `id_contrato + numero_parcela` (tolerância R$ 0,01); considerar a parcela quitada na última data de pagamento quando o acumulado cobre o previsto; calcular atraso na quitação ou contra a data de corte `max(data_real_pagamento)`, preservando os casos anômalos como atraso nulo (`<NA>`).
- **Resultado:** 1.659.408 linhas após deduplicação → 1.574.411 parcelas únicas após o filtro de versão → 122.867 contratos com cronograma; 7 atrasos nulos preservados.

### Decisão: Targets por Contrato (`OVER x MOB`) e Roll Rate

- **Problema:** no nível de parcela (1,57M chaves), a taxa 30+ parece baixa por diluição das parcelas em dia; já forçar os 36.359 contratos recusados (sem parcelas) como `target = 1` infla a taxa por contrato de 16,57% para 35,64% (30+) e de 10,12% para 30,67% (60+), enviesando o modelo em produção.
- **Regra:** consolidar no nível de `id_contrato`, com data de corte `max(data_real_pagamento)`, origem em `data_decisao` e janela MOB 12 de 360 dias. Contrato maduro é aquele cuja janela termina até a data de corte. Adotar **`OVER60MOB12`** (variante "fora") como target oficial: positivo apenas se o evento 60+ ocorrer em contrato maduro; zero para maduro sem incerteza; demais casos `<NA>`.
- **Resultado:** `OVER60MOB12` com 104.157 contratos avaliáveis (96.711 bons e 7.446 maus), taxa de **7,15%**, e 18.710 indeterminados. Foram descartados 930 positivos de janela imatura. Na variante "dentro" seriam 8.376 maus (7,97%), que foi descartada para não misturar safras com graus de observação diferentes.
- **Roll Rate:** a rolagem evoluiu de forma uniforme entre as coortes MOB 6, 9 e 12 (30+ → 60+ entre 57,2% e 57,8%; 60+ → 90+ entre 55,2% e 55,5%; 30+ → 90+ entre 31,7% e 31,9%), o que indica comportamento estável do atraso ao longo da janela de observação e sustenta o uso do 60+ como definição de inadimplência.